# QCHAN v4.0.0 — Corrected cohort extraction and empirical validation

This notebook advances the accepted QCHAN analytical preflight into the corrected 519-recording cohort stage. It follows the same governed architecture used for QGAIN, QADD, and QREV: the ten-domain checklist, G1–G10 gates, standardized Panels A–J with explicit N/A handling, support-aware non-imputed ML export, and separate immutable measurement/figure freezes.

**Scientific contract.** QCHAN measures reference-relative speech-spectrum deviation and upper-band attenuation proxies from guarded `strict_speech / primary` intervals. It does not identify a device, microphone, browser, platform, codec, or pure transfer function. The task-matched leave-one-subject-out reference is subject-balanced and has no global or cross-task fallback. Native source bandwidth, target support, reference membership, and reference vintage accompany every value.

This notebook is candidate-only. It does not publish or freeze QCHAN and cannot make final G10 retain/revise/drop decisions.


In [ ]:
from __future__ import annotations

from dataclasses import replace
from datetime import datetime, timezone
from hashlib import sha256
from pathlib import Path
import ast
import json
import os
import shutil
import subprocess
import sys
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import signal, stats
from IPython.display import display, Markdown


def find_project_root() -> Path:
    override = os.environ.get("PAPER1_PROJECT_ROOT", "").strip()
    if override:
        candidate = Path(override).expanduser().resolve()
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
        raise FileNotFoundError(f"PAPER1_PROJECT_ROOT is invalid: {candidate}")
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "src").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Open this notebook from inside the Paper 1 repository, "
        "or set PAPER1_PROJECT_ROOT."
    )


ROOT = find_project_root()
REVIEWED_SRC = ROOT / "src"
ORIGINAL_SRC = ROOT / "src"
for path in [REVIEWED_SRC, ORIGINAL_SRC]:
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))

from paper1_qc.media import decode_audio_views
from paper1_qc_reviewed.qchan_v400 import (
    ANALYSIS_FEATURES,
    DEFAULT_PARAMETERS,
    MEASUREMENT_VERSION,
    QChanParameters,
    TimeInterval,
    analysis_waveform_from_audio_views,
    build_subject_balanced_loso_references,
    compute_reference_relative_features,
    extract_recording_spectrum,
    feature_registry_frame,
    smoothed_log_ltas_db,
)
from paper1_qc_reviewed.qchan_v400_cohort import (
    CANONICAL_PROFILE,
    CANONICAL_STRICT_VIEW,
    COHORT_ORCHESTRATION_VERSION,
    ONE_SIDED_FEATURES,
    SIGNED_PRECURSORS,
    as_bool,
    canonical_interval_contract,
    date_column_for,
    deterministic_stratified_sample,
    deterministic_gallery_selection,
    gallery_linked_view_source,
    REQUIRED_GALLERY_LINKED_VIEWS,
    empirical_feature_summary,
    hash_inventory,
    intervals_for_recording,
    json_safe,
    load_recording_spectrum,
    ltas_source_frame,
    media_hash_column,
    model_interface_frame,
    native_bandwidth_summary,
    pairwise_redundancy,
    participant_balanced_resampling,
    participant_balanced_summary,
    reference_inventory_frame,
    reference_robustness_grid,
    remove_global_dc,
    repeated_recording_persistence,
    resolve_media_path,
    save_recording_spectrum,
    save_reference_spectrum,
    sha256_file,
    spectrum_checkpoint_complete,
    spectrum_checkpoint_paths,
    status_missingness_summary,
    subject_column_for,
    summarize_reference_robustness,
    support_availability_summary,
    task_stratum_series,
    unique_reference_frame,
    write_json,
)

# ---------------------------- execution controls ----------------------------
RUN_PACKAGE_TESTS = True
RUN_COHORT_EXTRACTION = False  # installer changes this to True in the local run copy
VERIFY_MEDIA_HASHES = True
REBUILD_CHECKPOINTS = False
RUN_TARGET_ROBUSTNESS = True
RUN_REFERENCE_ROBUSTNESS = True
BUILD_GALLERY = True

MAX_TARGET_ROBUSTNESS_RECORDINGS = 72
REFERENCE_ROBUSTNESS_TARGETS = 12
REFERENCE_BOOTSTRAP_ITERATIONS = 100
MAX_DELETE_REFERENCE_SUBJECTS = 16
PARTICIPANT_BALANCED_ITERATIONS = 1000
GALLERY_RECORDING_LIMIT = 10

MEDIA_ROOT_OVERRIDE = None  # installer inserts the local Bamboo_passage_only path
MEDIA_PATH_MAP = {}

PUBLISH_AND_FREEZE = False
SCIENTIFIC_REVIEW_DECISION = "PENDING"
SCIENTIFIC_REVIEWER = ""
SCIENTIFIC_REVIEW_RATIONALE = ""

PARAMETERS = DEFAULT_PARAMETERS
FS = PARAMETERS.analysis_sample_rate_hz
CANDIDATE_DIRNAME = "qchan-v4.0.0-candidate"

LEGACY_MAIN = ROOT / "MAIN outputs"
STAGE = ROOT / "MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates" / "channel_device" / CANDIDATE_DIRNAME
TABLES = STAGE / "tables"
LEDGERS = STAGE / "ledgers"
REFERENCES = STAGE / "references"
VALIDATION = STAGE / "validation"
FIGURES = STAGE / "figures"
GALLERIES = STAGE / "galleries"
AUDIT = STAGE / "audit"
MANIFESTS = STAGE / "manifests"
CHECKPOINTS = STAGE / "checkpoints"
for directory in [
    TABLES, LEDGERS, REFERENCES, VALIDATION, FIGURES, GALLERIES,
    AUDIT, MANIFESTS, CHECKPOINTS,
]:
    directory.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("default")
pd.set_option("display.max_columns", 240)
pd.set_option("display.width", 260)

print("Project root:", ROOT)
print("Measurement:", MEASUREMENT_VERSION)
print("Orchestration:", COHORT_ORCHESTRATION_VERSION)
print("Run cohort:", RUN_COHORT_EXTRACTION)
print("Stage:", STAGE)


In [ ]:
def save_table(
    frame: pd.DataFrame,
    path_without_suffix: Path,
    *,
    parquet: bool = True,
) -> dict:
    path_without_suffix = Path(path_without_suffix)
    path_without_suffix.parent.mkdir(parents=True, exist_ok=True)
    csv_path = path_without_suffix.with_suffix(".csv")
    frame.to_csv(csv_path, index=False)
    result = {"csv": str(csv_path), "csv_sha256": sha256_file(csv_path)}
    if parquet:
        parquet_path = path_without_suffix.with_suffix(".parquet")
        try:
            frame.to_parquet(parquet_path, index=False)
            result.update(
                {
                    "parquet": str(parquet_path),
                    "parquet_sha256": sha256_file(parquet_path),
                }
            )
        except Exception as exc:
            result["parquet_error"] = f"{type(exc).__name__}: {exc}"
            print(
                f"Parquet not written for {path_without_suffix.name}: "
                f"{type(exc).__name__}: {exc}"
            )
    return result


def save_figure_bundle(
    figure,
    *,
    stem: str,
    panel: str,
    source_data: pd.DataFrame,
    caption: str,
    scientific_question: str,
    alt_text: str,
    provenance: dict | None = None,
    destination: Path | None = None,
) -> dict:
    destination = FIGURES if destination is None else Path(destination)
    destination.mkdir(parents=True, exist_ok=True)
    png = destination / f"{stem}.png"
    svg = destination / f"{stem}.svg"
    pdf = destination / f"{stem}.pdf"
    source_csv = destination / f"{stem}.source.csv"
    caption_path = destination / f"{stem}.caption.md"
    provenance_path = destination / f"{stem}.provenance.json"

    figure.savefig(png, dpi=300, bbox_inches="tight")
    figure.savefig(svg, bbox_inches="tight")
    figure.savefig(pdf, bbox_inches="tight")
    plt.close(figure)

    source_data.to_csv(source_csv, index=False)
    caption_path.write_text(caption.strip() + "\n", encoding="utf-8")
    payload = {
        "panel": panel,
        "stem": stem,
        "measurement_version": MEASUREMENT_VERSION,
        "candidate_directory": CANDIDATE_DIRNAME,
        "scientific_question": scientific_question,
        "alt_text": alt_text,
        "source_csv": source_csv.name,
        "source_csv_sha256": sha256_file(source_csv),
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "feature_values_recomputed_by_figure": False,
        "selection_uses_diagnosis_or_human_qc": False,
        **dict(provenance or {}),
    }
    write_json(payload, provenance_path)
    return {
        "panel": panel,
        "stem": stem,
        "png": str(png.relative_to(STAGE)),
        "svg": str(svg.relative_to(STAGE)),
        "pdf": str(pdf.relative_to(STAGE)),
        "source_csv": str(source_csv.relative_to(STAGE)),
        "caption": str(caption_path.relative_to(STAGE)),
        "provenance": str(provenance_path.relative_to(STAGE)),
    }


def feature_unit(feature: str) -> str:
    return {
        "qchan_ltas_distance_db": "dB RMS",
        "qchan_rolloff95_deficit_hz": "Hz",
        "qchan_highband_ratio_deficit": "proportion",
        "qchan_tilt_steepening_db_per_oct": "dB/octave",
    }[feature]


def feature_label(feature: str) -> str:
    return {
        "qchan_ltas_distance_db": "LTAS distance",
        "qchan_rolloff95_deficit_hz": "Rolloff-95 deficit",
        "qchan_highband_ratio_deficit": "High-band deficit",
        "qchan_tilt_steepening_db_per_oct": "Tilt steepening",
    }[feature]


def finite_spearman(left, right) -> tuple[int, float]:
    pair = pd.DataFrame(
        {
            "left": pd.to_numeric(left, errors="coerce"),
            "right": pd.to_numeric(right, errors="coerce"),
        }
    ).dropna()
    rho = (
        float(stats.spearmanr(pair["left"], pair["right"]).statistic)
        if len(pair) >= 3
        and pair["left"].nunique() > 1
        and pair["right"].nunique() > 1
        else np.nan
    )
    return len(pair), rho


def parameter_sensitivity_specifications() -> list[tuple[str, QChanParameters]]:
    """Return technically resolvable QCHAN estimator variants.

    The frozen 16-kHz PSD grid uses n_fft=2048 (7.8125-Hz spacing). A
    one-sixth-octave grid beginning at 100 Hz contains bands narrower than the
    saved grid can support and is therefore not a valid sensitivity condition.
    The smoothing sensitivity grid uses the baseline one-third-octave bands
    together with resolvable one-half- and one-octave alternatives.
    """
    return [
        ("baseline", PARAMETERS),
        (
            "floor_-100dB",
            replace(PARAMETERS, relative_psd_floor_db=-100.0),
        ),
        (
            "floor_-60dB",
            replace(PARAMETERS, relative_psd_floor_db=-60.0),
        ),
        (
            "analysis_high_7000Hz",
            replace(
                PARAMETERS,
                analysis_high_hz=7000.0,
                highband_high_hz=7000.0,
            ),
        ),
        (
            "highband_low_2500Hz",
            replace(PARAMETERS, highband_low_hz=2500.0),
        ),
        (
            "highband_low_3500Hz",
            replace(PARAMETERS, highband_low_hz=3500.0),
        ),
        (
            "rolloff_90pct",
            replace(PARAMETERS, rolloff_fraction=0.90),
        ),
        (
            "rolloff_99pct",
            replace(PARAMETERS, rolloff_fraction=0.99),
        ),
        (
            "octave_fraction_1",
            replace(PARAMETERS, octave_fraction=1),
        ),
        (
            "octave_fraction_2",
            replace(PARAMETERS, octave_fraction=2),
        ),
        (
            "tilt_high_3000Hz",
            replace(PARAMETERS, tilt_high_hz=3000.0),
        ),
        (
            "tilt_high_5000Hz",
            replace(PARAMETERS, tilt_high_hz=5000.0),
        ),
    ]


package_test_output = ""
package_tests_passed = False
if RUN_PACKAGE_TESTS:
    completed = subprocess.run(
        [
            sys.executable,
            "-m",
            "pytest",
            str(ROOT / "tests" / "test_qchan_v400.py"),
            str(ROOT / "tests" / "test_qchan_v400_cohort.py"),
            "-q",
            "--disable-warnings",
        ],
        cwd=ROOT,
        text=True,
        capture_output=True,
    )
    package_test_output = completed.stdout + completed.stderr
    print(package_test_output)
    package_tests_passed = completed.returncode == 0
    if not package_tests_passed:
        raise RuntimeError("Reviewed QCHAN package tests failed")
else:
    package_tests_passed = True

figure_index_rows: list[dict] = []


In [ ]:
# Verify the accepted QCHAN preflight and complete A-C artifact bundles.
preflight_manifest_path = MANIFESTS / "qchan_v400_preflight_manifest.json"
if not preflight_manifest_path.exists():
    raise FileNotFoundError(
        "The accepted QCHAN v4.0.0 preflight manifest is missing. "
        "Run and save the reviewed preflight notebook first."
    )
preflight_manifest = json.loads(
    preflight_manifest_path.read_text(encoding="utf-8")
)
preflight_checks = pd.read_csv(TABLES / "qchan_v400_preflight_all_checks.csv")
preflight_gates = pd.read_csv(TABLES / "qchan_v400_gate_summary.csv")
preflight_figure_index = pd.read_csv(
    TABLES / "qchan_v400_preflight_figure_index.csv"
)

preflight_bundle_rows = []
for panel, stem in [
    ("A", "A_construct_response"),
    ("B", "B_discriminant_specificity"),
    ("C", "C_transformation_contract"),
]:
    expected = [
        FIGURES / f"{stem}.png",
        FIGURES / f"{stem}.svg",
        FIGURES / f"{stem}.pdf",
        FIGURES / f"{stem}.source.csv",
        FIGURES / f"{stem}.caption.md",
        FIGURES / f"{stem}.provenance.json",
    ]
    preflight_bundle_rows.append(
        {
            "panel": panel,
            "stem": stem,
            "complete": all(path.exists() for path in expected),
            "artifact_count": sum(path.exists() for path in expected),
        }
    )
    figure_index_rows.append(
        {
            "panel": panel,
            "stem": stem,
            "png": f"figures/{stem}.png",
            "svg": f"figures/{stem}.svg",
            "pdf": f"figures/{stem}.pdf",
            "source_csv": f"figures/{stem}.source.csv",
            "caption": f"figures/{stem}.caption.md",
            "provenance": f"figures/{stem}.provenance.json",
        }
    )
preflight_bundle_check = pd.DataFrame(preflight_bundle_rows)

accepted_preflight = bool(
    preflight_manifest.get("preflight_blocking_checks_pass")
    and preflight_manifest.get("package_tests_passed")
    and not preflight_manifest.get("cohort_extraction_completed")
    and not preflight_manifest.get("freeze_allowed")
    and preflight_checks["passed"].astype(bool).all()
    and preflight_bundle_check["complete"].all()
)
if not accepted_preflight:
    display(preflight_bundle_check)
    display(preflight_checks)
    raise RuntimeError("Accepted QCHAN preflight contract failed")

save_table(
    preflight_bundle_check,
    AUDIT / "qchan_v400_accepted_preflight_bundles",
    parquet=False,
)
print("Accepted QCHAN preflight and Panels A-C verified.")
display(preflight_bundle_check)


In [ ]:
# Load frozen cohort inputs and enforce the exact strict-speech contract.
DATA_FREEZE = LEGACY_MAIN / "00_DATA_FREEZE" / "v1"
SEGMENTATION_FREEZE = LEGACY_MAIN / "01_SEGMENTATION_FREEZE" / "v1"
recordings_path = DATA_FREEZE / "frozen_bamboo_recordings.csv"
decisions_path = SEGMENTATION_FREEZE / "frozen_segmentation_decisions.csv"
intervals_path = SEGMENTATION_FREEZE / "frozen_segmentation_intervals.csv"
for path in [recordings_path, decisions_path, intervals_path]:
    if not path.exists():
        raise FileNotFoundError(path)

frozen_recordings = pd.read_csv(recordings_path, low_memory=False)
frozen_decisions = pd.read_csv(decisions_path, low_memory=False)
frozen_intervals = pd.read_csv(intervals_path, low_memory=False)

recording_eligible = as_bool(frozen_recordings["freeze_included"])
segmentation_eligible = as_bool(
    frozen_decisions["segmentation_analysis_eligible"]
)
frozen = frozen_recordings.loc[recording_eligible].merge(
    frozen_decisions.loc[
        segmentation_eligible,
        [
            "logical_recording_id",
            "segmentation_analysis_eligible",
            "segmentation_decision_source",
        ],
    ],
    on="logical_recording_id",
    how="inner",
    validate="one_to_one",
)
frozen["logical_recording_id"] = frozen["logical_recording_id"].astype(str)
frozen = frozen.sort_values("logical_recording_id").reset_index(drop=True)

strict_table, interval_contract = canonical_interval_contract(
    frozen_decisions, frozen_intervals
)
strict_table = strict_table.loc[
    strict_table["logical_recording_id"].isin(frozen["logical_recording_id"])
].copy()

subject_column = subject_column_for(frozen)
date_column = date_column_for(frozen)
media_sha_column = media_hash_column(frozen)
task_series, task_source = task_stratum_series(frozen)
frozen["qchan_subject_id"] = frozen[subject_column].astype(str)
frozen["qchan_task_stratum"] = task_series.astype(str)

reference_metadata = frozen[
    ["logical_recording_id", "qchan_subject_id", "qchan_task_stratum"]
].rename(
    columns={
        "qchan_subject_id": "subject_id",
        "qchan_task_stratum": "task_stratum",
    }
)

input_artifacts = pd.DataFrame(
    [
        {
            "artifact": "frozen_bamboo_recordings.csv",
            "path": str(recordings_path),
            "sha256": sha256_file(recordings_path),
        },
        {
            "artifact": "frozen_segmentation_decisions.csv",
            "path": str(decisions_path),
            "sha256": sha256_file(decisions_path),
        },
        {
            "artifact": "frozen_segmentation_intervals.csv",
            "path": str(intervals_path),
            "sha256": sha256_file(intervals_path),
        },
        {
            "artifact": "qchan_v400.py",
            "path": str(
                REVIEWED_SRC / "paper1_qc_reviewed" / "qchan_v400.py"
            ),
            "sha256": sha256_file(
                REVIEWED_SRC / "paper1_qc_reviewed" / "qchan_v400.py"
            ),
        },
        {
            "artifact": "qchan_v400_cohort.py",
            "path": str(
                REVIEWED_SRC
                / "paper1_qc_reviewed"
                / "qchan_v400_cohort.py"
            ),
            "sha256": sha256_file(
                REVIEWED_SRC
                / "paper1_qc_reviewed"
                / "qchan_v400_cohort.py"
            ),
        },
    ]
)
save_table(input_artifacts, AUDIT / "qchan_v400_input_artifacts", parquet=False)
save_table(
    interval_contract,
    AUDIT / "qchan_v400_canonical_interval_contract",
    parquet=False,
)
save_table(
    strict_table,
    LEDGERS / "qchan_v400_canonical_strict_speech_intervals",
)
save_table(feature_registry_frame(), TABLES / "qchan_v400_feature_registry", parquet=False)
save_table(reference_metadata, LEDGERS / "qchan_v400_reference_metadata", parquet=False)

input_checks = pd.DataFrame(
    [
        {
            "gate": "G1",
            "check": "corrected cohort count",
            "passed": len(frozen) == 519,
            "observed": len(frozen),
            "required": 519,
        },
        {
            "gate": "G1",
            "check": "participant count",
            "passed": frozen[subject_column].nunique(dropna=True) == 224,
            "observed": frozen[subject_column].nunique(dropna=True),
            "required": 224,
        },
        {
            "gate": "G1",
            "check": "canonical strict-speech interval contract",
            "passed": interval_contract["contract_pass"].astype(bool).all(),
            "observed": interval_contract["contract_pass"].tolist(),
            "required": "all true",
        },
        {
            "gate": "G1",
            "check": "all eligible recordings have strict intervals",
            "passed": strict_table["logical_recording_id"].nunique()
            == len(frozen),
            "observed": strict_table["logical_recording_id"].nunique(),
            "required": len(frozen),
        },
        {
            "gate": "G1",
            "check": "task matching source declared",
            "passed": bool(task_source),
            "observed": task_source,
            "required": "frozen task field or declared Bamboo constant",
        },
        {
            "gate": "G1",
            "check": "no clinical labels in reference metadata",
            "passed": set(reference_metadata.columns)
            == {"logical_recording_id", "subject_id", "task_stratum"},
            "observed": list(reference_metadata.columns),
            "required": [
                "logical_recording_id",
                "subject_id",
                "task_stratum",
            ],
        },
    ]
)
if not input_checks["passed"].astype(bool).all():
    display(input_checks)
    raise RuntimeError("Frozen QCHAN cohort contract failed")
save_table(input_checks, VALIDATION / "qchan_v400_g1_input_checks", parquet=False)

display(input_checks)
display(interval_contract)


In [ ]:
# Recording-level spectrum extraction with restart-safe checkpoints.
ffmpeg = shutil.which("ffmpeg")
ffprobe = shutil.which("ffprobe")
if RUN_COHORT_EXTRACTION and (not ffmpeg or not ffprobe):
    raise RuntimeError("ffmpeg and ffprobe must be available on PATH")

if REBUILD_CHECKPOINTS and CHECKPOINTS.exists():
    shutil.rmtree(CHECKPOINTS)
for directory in [
    CHECKPOINTS / "spectra",
    CHECKPOINTS / "records",
    CHECKPOINTS / "media_audit",
    CHECKPOINTS / "errors",
]:
    directory.mkdir(parents=True, exist_ok=True)

spectra = {}
spectrum_rows = []
media_audit_rows = []
error_rows = []

if RUN_COHORT_EXTRACTION:
    started = time.time()
    for position, (_, frozen_row) in enumerate(
        frozen.iterrows(), start=1
    ):
        recording_id = str(frozen_row["logical_recording_id"])
        paths = spectrum_checkpoint_paths(CHECKPOINTS, recording_id)
        try:
            if spectrum_checkpoint_complete(paths) and not REBUILD_CHECKPOINTS:
                spectrum = load_recording_spectrum(paths["spectrum"])
                record = json.loads(
                    paths["metadata"].read_text(encoding="utf-8")
                )
                media_audit = json.loads(
                    paths["media"].read_text(encoding="utf-8")
                )
                spectra[recording_id] = spectrum
                spectrum_rows.append(record)
                media_audit_rows.append(media_audit)
                continue

            media_path = resolve_media_path(
                frozen_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE)
                    if MEDIA_ROOT_OVERRIDE
                    else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            observed_sha = (
                sha256_file(media_path)
                if VERIFY_MEDIA_HASHES
                else "not_checked"
            )
            if media_sha_column and not pd.isna(frozen_row[media_sha_column]):
                expected_sha = str(frozen_row[media_sha_column]).strip()
            else:
                expected_sha = ""
            hash_match = bool(
                not VERIFY_MEDIA_HASHES
                or not expected_sha
                or observed_sha == expected_sha
            )
            if not hash_match:
                raise RuntimeError(
                    f"Media SHA-256 mismatch for {recording_id}: "
                    f"expected {expected_sha}, observed {observed_sha}"
                )

            views = decode_audio_views(
                media_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=FS,
            )
            waveform_raw = analysis_waveform_from_audio_views(views)
            waveform, dc_before, dc_after = remove_global_dc(waveform_raw)
            strict, strict_rows = intervals_for_recording(
                strict_table, recording_id
            )
            spectrum = extract_recording_spectrum(
                waveform,
                FS,
                strict_speech=strict,
                logical_recording_id=recording_id,
                source_sample_rate_hz=int(views.sample_rate_native),
                parameters=PARAMETERS,
            )

            probe = dict(views.probe or {})
            record = {
                "logical_recording_id": recording_id,
                "qchan_measurement_version": MEASUREMENT_VERSION,
                "qchan_spectrum_status": spectrum.status,
                "qchan_support_tier": spectrum.support_tier,
                "qchan_guarded_speech_support_sec": (
                    spectrum.guarded_speech_support_sec
                ),
                "qchan_valid_frame_count": spectrum.valid_frame_count,
                "qchan_guarded_segment_count": spectrum.guarded_segment_count,
                "qchan_zero_frame_count": spectrum.zero_frame_count,
                "qchan_source_sample_rate_hz": spectrum.source_sample_rate_hz,
                "qchan_source_nyquist_hz": spectrum.source_nyquist_hz,
                "qchan_source_bandwidth_limited": (
                    spectrum.source_bandwidth_limited
                ),
                "qchan_spectrum_sha256": spectrum.spectrum_sha256,
                "qchan_analysis_waveform_field": "analysis_16k",
                "qchan_global_dc_removal_applied": True,
                "qchan_waveform_mean_before_dc_removal": dc_before,
                "qchan_waveform_mean_after_dc_removal": dc_after,
                "qchan_analysis_duration_sec": len(waveform) / FS,
                "qchan_strict_interval_count": len(strict_rows),
                "qchan_media_path_resolved": str(media_path),
                "qchan_media_sha256_observed": observed_sha,
                "qchan_media_sha256_match": hash_match,
                "qchan_native_channels": probe.get("channels"),
                "qchan_native_channel_layout": probe.get("channel_layout"),
                "qchan_native_codec_name": probe.get("codec_name"),
                "qchan_native_container_format": probe.get("container_format"),
                "qchan_decoder": str(ffmpeg),
                "qchan_ffprobe": str(ffprobe),
                "qchan_decode_warning": str(views.decode_stderr or ""),
            }

            media_audit = {
                "logical_recording_id": recording_id,
                "media_path": str(media_path),
                "expected_sha256": expected_sha,
                "observed_sha256": observed_sha,
                "sha256_match": hash_match,
                "analysis_sample_rate_hz": FS,
                "analysis_duration_sec": len(waveform) / FS,
                "analysis_waveform_field": "analysis_16k",
                "global_dc_removal_applied": True,
                "native_sample_rate_hz": int(views.sample_rate_native),
                "native_channels": probe.get("channels"),
                "native_channel_layout": probe.get("channel_layout"),
                "native_codec_name": probe.get("codec_name"),
                "native_container_format": probe.get("container_format"),
            }

            save_recording_spectrum(spectrum, paths["spectrum"])
            paths["metadata"].write_text(
                json.dumps(json_safe(record), indent=2),
                encoding="utf-8",
            )
            paths["media"].write_text(
                json.dumps(json_safe(media_audit), indent=2),
                encoding="utf-8",
            )
            if paths["error"].exists():
                paths["error"].unlink()

            spectra[recording_id] = spectrum
            spectrum_rows.append(record)
            media_audit_rows.append(media_audit)

            if position == 1 or position % 20 == 0 or position == len(frozen):
                elapsed = time.time() - started
                rate = elapsed / position
                remaining_min = rate * (len(frozen) - position) / 60.0
                print(
                    f"[{position:03d}/{len(frozen)}] {recording_id} "
                    f"| ETA {remaining_min:.1f} min"
                )
        except Exception as exc:
            error = {
                "logical_recording_id": recording_id,
                "error_type": type(exc).__name__,
                "message": str(exc),
            }
            error_rows.append(error)
            paths["error"].write_text(
                json.dumps(error, indent=2), encoding="utf-8"
            )
            print("ERROR", recording_id, type(exc).__name__, exc)

spectrum_table = pd.DataFrame(spectrum_rows)
media_audit = pd.DataFrame(media_audit_rows)
extraction_errors = pd.DataFrame(
    error_rows,
    columns=["logical_recording_id", "error_type", "message"],
)

save_table(spectrum_table, TABLES / "qchan_v400_recording_spectrum_manifest")
save_table(media_audit, AUDIT / "qchan_v400_media_hash_audit", parquet=False)
save_table(
    extraction_errors,
    AUDIT / "qchan_v400_extraction_errors",
    parquet=False,
)

extraction_checks = pd.DataFrame(
    [
        {
            "gate": "G7",
            "check": "all frozen recordings have spectrum checkpoints",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or len(spectrum_table) == len(frozen)
            ),
            "observed": len(spectrum_table),
            "required": len(frozen),
        },
        {
            "gate": "G7",
            "check": "no spectrum extraction errors",
            "passed": bool(
                not RUN_COHORT_EXTRACTION or extraction_errors.empty
            ),
            "observed": len(extraction_errors),
            "required": 0,
        },
        {
            "gate": "G7",
            "check": "all media hashes verified",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(media_audit) == len(frozen)
                    and media_audit["sha256_match"].astype(bool).all()
                )
            ),
            "observed": (
                int(media_audit["sha256_match"].astype(bool).sum())
                if len(media_audit)
                else 0
            ),
            "required": len(frozen),
        },
        {
            "gate": "G3",
            "check": "analysis_16k and global DC contract enforced",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    spectrum_table["qchan_analysis_waveform_field"]
                    .astype(str)
                    .eq("analysis_16k")
                    .all()
                    and spectrum_table[
                        "qchan_global_dc_removal_applied"
                    ]
                    .astype(bool)
                    .all()
                    and pd.to_numeric(
                        spectrum_table[
                            "qchan_waveform_mean_after_dc_removal"
                        ],
                        errors="coerce",
                    )
                    .abs()
                    .max()
                    <= 1e-10
                )
            ),
            "observed": (
                float(
                    pd.to_numeric(
                        spectrum_table[
                            "qchan_waveform_mean_after_dc_removal"
                        ],
                        errors="coerce",
                    )
                    .abs()
                    .max()
                )
                if len(spectrum_table)
                else np.nan
            ),
            "required": "<=1e-10",
        },
    ]
)
save_table(
    extraction_checks,
    VALIDATION / "qchan_v400_extraction_checks",
    parquet=False,
)
display(extraction_checks)
display(spectrum_table.head())

if RUN_COHORT_EXTRACTION and not extraction_checks["passed"].astype(bool).all():
    raise RuntimeError(
        "QCHAN spectrum extraction did not satisfy the frozen cohort contract."
    )


In [ ]:
# Build task-matched subject-balanced LOSO references and compute four features.
references = {}
recording_rows = []
reference_errors = []

if RUN_COHORT_EXTRACTION and len(spectra) == len(frozen):
    references = build_subject_balanced_loso_references(
        spectra,
        reference_metadata,
        PARAMETERS,
    )

    unique_saved = set()
    for recording_id, reference in references.items():
        if reference.reference_key not in unique_saved:
            save_reference_spectrum(
                reference,
                REFERENCES / f"{reference.reference_key}.npz",
            )
            unique_saved.add(reference.reference_key)

    frozen_lookup = frozen.set_index("logical_recording_id")
    for recording_id in frozen["logical_recording_id"].astype(str):
        try:
            result = compute_reference_relative_features(
                spectra[recording_id],
                references[recording_id],
                PARAMETERS,
            )
            source_row = frozen_lookup.loc[recording_id]
            for column in [
                subject_column,
                date_column,
                "diagnosis_analysis",
                "selected_media_file_name",
                "selected_media_extension",
                "media_path",
                "media_sha256",
            ]:
                if column in source_row.index:
                    result[column] = source_row[column]
            result["qchan_task_stratum"] = source_row[
                "qchan_task_stratum"
            ]
            recording_rows.append(result)
        except Exception as exc:
            reference_errors.append(
                {
                    "logical_recording_id": recording_id,
                    "error_type": type(exc).__name__,
                    "message": str(exc),
                }
            )

recording_features = pd.DataFrame(recording_rows)
reference_error_table = pd.DataFrame(
    reference_errors,
    columns=["logical_recording_id", "error_type", "message"],
)
reference_inventory = (
    reference_inventory_frame(references, reference_metadata)
    if references
    else pd.DataFrame()
)
unique_references = (
    unique_reference_frame(references) if references else pd.DataFrame()
)

if RUN_COHORT_EXTRACTION:
    if recording_features["logical_recording_id"].duplicated().any():
        raise RuntimeError("Recording-level QCHAN feature rows are duplicated.")
    recording_table = frozen.merge(
        recording_features,
        on="logical_recording_id",
        how="left",
        suffixes=("", "__qchan"),
        validate="one_to_one",
    )
else:
    recording_table = pd.DataFrame()

save_table(recording_table, TABLES / "qchan_v400_analysis_features")
save_table(
    reference_inventory,
    LEDGERS / "qchan_v400_reference_ledger",
)
save_table(
    unique_references,
    LEDGERS / "qchan_v400_unique_reference_inventory",
)
save_table(
    reference_error_table,
    AUDIT / "qchan_v400_reference_errors",
    parquet=False,
)

reference_checks = pd.DataFrame(
    [
        {
            "gate": "G6",
            "check": "reference row for every frozen recording",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or len(reference_inventory) == len(frozen)
            ),
            "observed": len(reference_inventory),
            "required": len(frozen),
        },
        {
            "gate": "G6",
            "check": "target subject excluded from every reference",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(reference_inventory) == len(frozen)
                    and reference_inventory[
                        "target_subject_excluded"
                    ]
                    .astype(bool)
                    .all()
                )
            ),
            "observed": (
                int(
                    reference_inventory[
                        "target_subject_excluded"
                    ]
                    .astype(bool)
                    .sum()
                )
                if len(reference_inventory)
                else 0
            ),
            "required": len(frozen),
        },
        {
            "gate": "G6",
            "check": "reference support rule or explicit unavailable state",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    (
                        reference_inventory[
                            "reference_status"
                        ].astype(str).eq("measured")
                        & reference_inventory[
                            "reference_subject_count"
                        ].ge(PARAMETERS.minimum_reference_subjects)
                        & reference_inventory[
                            "reference_recording_count"
                        ].ge(PARAMETERS.minimum_reference_recordings)
                    )
                    | reference_inventory[
                        "reference_status"
                    ].astype(str).eq("reference_unavailable")
                ).all()
            ),
            "observed": (
                reference_inventory[
                    "reference_status"
                ].value_counts().to_dict()
                if len(reference_inventory)
                else {}
            ),
            "required": "measured with >=5 subjects/>=8 recordings or explicit unavailable",
        },
        {
            "gate": "G7",
            "check": "four-feature row for every frozen recording",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or len(recording_table) == len(frozen)
            ),
            "observed": len(recording_table),
            "required": len(frozen),
        },
        {
            "gate": "G7",
            "check": "no reference/feature errors",
            "passed": bool(
                not RUN_COHORT_EXTRACTION or reference_error_table.empty
            ),
            "observed": len(reference_error_table),
            "required": 0,
        },
    ]
)
save_table(
    reference_checks,
    VALIDATION / "qchan_v400_reference_checks",
    parquet=False,
)
display(reference_checks)
display(recording_table.head())
display(reference_inventory.head())

if RUN_COHORT_EXTRACTION and not reference_checks["passed"].astype(bool).all():
    raise RuntimeError("QCHAN reference construction or feature extraction failed.")


In [ ]:
# G2 — reconstruct every recording-level feature from saved spectra/references.
reconstruction_rows = []
if RUN_COHORT_EXTRACTION and len(recording_table):
    feature_lookup = recording_features.set_index("logical_recording_id")
    for recording_id in frozen["logical_recording_id"].astype(str):
        recomputed = compute_reference_relative_features(
            spectra[recording_id],
            references[recording_id],
            PARAMETERS,
        )
        stored = feature_lookup.loc[recording_id]
        for feature in [
            *ANALYSIS_FEATURES,
            *SIGNED_PRECURSORS.values(),
        ]:
            stored_value = pd.to_numeric(
                pd.Series([stored.get(feature)]), errors="coerce"
            ).iloc[0]
            reconstructed_value = pd.to_numeric(
                pd.Series([recomputed.get(feature)]), errors="coerce"
            ).iloc[0]
            both_missing = pd.isna(stored_value) and pd.isna(
                reconstructed_value
            )
            exact = bool(
                both_missing
                or (
                    np.isfinite(stored_value)
                    and np.isfinite(reconstructed_value)
                    and abs(stored_value - reconstructed_value) <= 1e-12
                )
            )
            reconstruction_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "feature": feature,
                    "stored_value": stored_value,
                    "reconstructed_value": reconstructed_value,
                    "absolute_difference": (
                        abs(stored_value - reconstructed_value)
                        if np.isfinite(stored_value)
                        and np.isfinite(reconstructed_value)
                        else np.nan
                    ),
                    "exact_or_both_missing": exact,
                }
            )

reconstruction_audit = pd.DataFrame(reconstruction_rows)
save_table(
    reconstruction_audit,
    AUDIT / "qchan_v400_reconstruction_audit",
)

g2_checks = pd.DataFrame(
    [
        {
            "gate": "G2",
            "check": "recording features reconstruct from saved spectra and reference ledger",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(reconstruction_audit)
                    == len(frozen)
                    * (len(ANALYSIS_FEATURES) + len(SIGNED_PRECURSORS))
                    and reconstruction_audit[
                        "exact_or_both_missing"
                    ]
                    .astype(bool)
                    .all()
                )
            ),
            "observed": (
                int(
                    reconstruction_audit[
                        "exact_or_both_missing"
                    ]
                    .astype(bool)
                    .sum()
                )
                if len(reconstruction_audit)
                else 0
            ),
            "required": len(frozen)
            * (len(ANALYSIS_FEATURES) + len(SIGNED_PRECURSORS)),
        },
        {
            "gate": "G2",
            "check": "one-sided features are nonnegative when available",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or all(
                    pd.to_numeric(
                        recording_table[feature], errors="coerce"
                    )
                    .dropna()
                    .ge(-1e-12)
                    .all()
                    for feature in ONE_SIDED_FEATURES
                )
            ),
            "observed": {
                feature: float(
                    pd.to_numeric(
                        recording_table[feature], errors="coerce"
                    ).min()
                )
                for feature in ONE_SIDED_FEATURES
            }
            if len(recording_table)
            else {},
            "required": ">=0",
        },
        {
            "gate": "G2",
            "check": "unavailable features remain missing rather than zero-filled",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or all(
                    pd.to_numeric(
                        recording_table.loc[
                            ~recording_table[
                                f"{feature}_status"
                            ]
                            .astype(str)
                            .eq("measured"),
                            feature,
                        ],
                        errors="coerce",
                    )
                    .isna()
                    .all()
                    for feature in ANALYSIS_FEATURES
                )
            ),
            "observed": "status/value contract",
            "required": "all unavailable values missing",
        },
    ]
)
save_table(
    g2_checks,
    VALIDATION / "qchan_v400_g2_reconstruction_checks",
    parquet=False,
)
display(g2_checks)

if RUN_COHORT_EXTRACTION and not g2_checks["passed"].astype(bool).all():
    raise RuntimeError("QCHAN reconstruction checks failed.")


In [ ]:
# G6 — target-side window, guard, boundary, segment, and parameter sensitivity.
# R1 correction: isolate each variant, preserve explicit schemas, and use only
# smoothing grids that are resolvable on the frozen n_fft=2048 PSD grid.
TARGET_ROBUSTNESS_COLUMNS = [
    "logical_recording_id",
    "variant",
    "feature",
    "baseline_value",
    "variant_value",
    "baseline_available",
    "variant_available",
    "absolute_delta",
    "variant_spectrum_status",
    "variant_support_sec",
    "variant_valid_frame_count",
]
PARAMETER_SENSITIVITY_COLUMNS = [
    "logical_recording_id",
    "variant",
    "feature",
    "baseline_value",
    "variant_value",
    "baseline_available",
    "variant_available",
    "absolute_delta",
]
ROBUSTNESS_ERROR_COLUMNS = [
    "logical_recording_id",
    "stage",
    "variant",
    "error_type",
    "message",
]
SENSITIVITY_SUMMARY_COLUMNS = [
    "variant",
    "feature",
    "recording_count",
    "paired_finite_n",
    "availability_agreement",
    "spearman_rho",
    "median_absolute_delta",
    "p95_absolute_delta",
    "maximum_absolute_delta",
]

target_robustness_rows = []
parameter_sensitivity_rows = []
target_robustness_errors = []
target_robustness_sample = pd.DataFrame(
    columns=[
        "logical_recording_id",
        "qchan_support_tier",
        "qchan_source_bandwidth_limited",
        "qchan_ltas_distance_db",
        "ltas_quantile",
    ]
)

if (
    RUN_COHORT_EXTRACTION
    and RUN_TARGET_ROBUSTNESS
    and len(recording_table)
):
    sample_frame = recording_table[
        [
            "logical_recording_id",
            "qchan_support_tier",
            "qchan_source_bandwidth_limited",
            "qchan_ltas_distance_db",
        ]
    ].copy()
    numeric_ltas = pd.to_numeric(
        sample_frame["qchan_ltas_distance_db"], errors="coerce"
    )
    finite_unique = int(numeric_ltas.dropna().nunique())
    if finite_unique >= 2:
        sample_frame["ltas_quantile"] = pd.qcut(
            numeric_ltas,
            q=min(4, finite_unique),
            duplicates="drop",
        ).astype(str)
    else:
        sample_frame["ltas_quantile"] = np.where(
            numeric_ltas.notna(), "single_finite_stratum", "missing"
        )

    target_robustness_sample = deterministic_stratified_sample(
        sample_frame,
        maximum_rows=MAX_TARGET_ROBUSTNESS_RECORDINGS,
        stratum_columns=[
            "qchan_support_tier",
            "qchan_source_bandwidth_limited",
            "ltas_quantile",
        ],
    )
    save_table(
        target_robustness_sample,
        VALIDATION / "qchan_v400_target_robustness_sample",
        parquet=False,
    )

    frozen_lookup = frozen.set_index("logical_recording_id")
    feature_lookup = recording_features.set_index("logical_recording_id")

    target_specification_names = [
        "baseline",
        "frame_30ms",
        "frame_50ms",
        "hop_20ms",
        "guard_100ms",
        "guard_300ms",
        "boundary_erosion_50ms",
        "boundary_dilation_50ms",
        "delete_longest_strict_segment",
    ]
    parameter_specifications = parameter_sensitivity_specifications()

    for selected in target_robustness_sample.itertuples(index=False):
        recording_id = str(selected.logical_recording_id)

        # Decode/setup is isolated from variant computation so any failure is
        # explicit and does not later surface as a misleading missing-column error.
        try:
            source_row = frozen_lookup.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE)
                    if MEDIA_ROOT_OVERRIDE
                    else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(
                media_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=FS,
            )
            waveform, _, _ = remove_global_dc(
                analysis_waveform_from_audio_views(views)
            )
            duration_sec = len(waveform) / FS
            strict, _ = intervals_for_recording(
                strict_table, recording_id
            )
            baseline_reference = references[recording_id]
            baseline_values = feature_lookup.loc[recording_id].to_dict()
            baseline_observation = spectra[recording_id]
        except Exception as exc:
            target_robustness_errors.append(
                {
                    "logical_recording_id": recording_id,
                    "stage": "recording_setup",
                    "variant": "",
                    "error_type": type(exc).__name__,
                    "message": str(exc),
                }
            )
            continue

        def erode_intervals(intervals, erosion_ms):
            erosion = float(erosion_ms) / 1000.0
            result = []
            for interval in intervals:
                start = max(0.0, interval.start_sec + erosion)
                end = min(duration_sec, interval.end_sec - erosion)
                if end - start >= 0.05:
                    result.append(TimeInterval(start, end))
            return result

        longest_index = (
            int(
                np.argmax(
                    [
                        interval.end_sec - interval.start_sec
                        for interval in strict
                    ]
                )
            )
            if strict
            else -1
        )
        deleted = [
            interval
            for index, interval in enumerate(strict)
            if index != longest_index
        ]
        target_specifications = [
            ("baseline", PARAMETERS, strict),
            ("frame_30ms", replace(PARAMETERS, frame_ms=30.0), strict),
            ("frame_50ms", replace(PARAMETERS, frame_ms=50.0), strict),
            ("hop_20ms", replace(PARAMETERS, hop_ms=20.0), strict),
            (
                "guard_100ms",
                replace(PARAMETERS, speech_boundary_guard_ms=100.0),
                strict,
            ),
            (
                "guard_300ms",
                replace(PARAMETERS, speech_boundary_guard_ms=300.0),
                strict,
            ),
            (
                "boundary_erosion_50ms",
                PARAMETERS,
                erode_intervals(strict, 50.0),
            ),
            (
                "boundary_dilation_50ms",
                PARAMETERS,
                erode_intervals(strict, -50.0),
            ),
            (
                "delete_longest_strict_segment",
                PARAMETERS,
                deleted,
            ),
        ]

        for variant, parameters, variant_intervals in target_specifications:
            try:
                observation = extract_recording_spectrum(
                    waveform,
                    FS,
                    strict_speech=variant_intervals,
                    logical_recording_id=f"{recording_id}|{variant}",
                    source_sample_rate_hz=int(views.sample_rate_native),
                    parameters=parameters,
                )
                values = compute_reference_relative_features(
                    observation,
                    baseline_reference,
                    parameters,
                )
                for feature in ANALYSIS_FEATURES:
                    baseline_value = pd.to_numeric(
                        pd.Series([baseline_values.get(feature)]),
                        errors="coerce",
                    ).iloc[0]
                    variant_value = pd.to_numeric(
                        pd.Series([values.get(feature)]),
                        errors="coerce",
                    ).iloc[0]
                    target_robustness_rows.append(
                        {
                            "logical_recording_id": recording_id,
                            "variant": variant,
                            "feature": feature,
                            "baseline_value": baseline_value,
                            "variant_value": variant_value,
                            "baseline_available": bool(
                                np.isfinite(baseline_value)
                            ),
                            "variant_available": bool(
                                np.isfinite(variant_value)
                            ),
                            "absolute_delta": (
                                abs(variant_value - baseline_value)
                                if np.isfinite(variant_value)
                                and np.isfinite(baseline_value)
                                else np.nan
                            ),
                            "variant_spectrum_status": observation.status,
                            "variant_support_sec": (
                                observation.guarded_speech_support_sec
                            ),
                            "variant_valid_frame_count": (
                                observation.valid_frame_count
                            ),
                        }
                    )
            except Exception as exc:
                target_robustness_errors.append(
                    {
                        "logical_recording_id": recording_id,
                        "stage": "target_variant",
                        "variant": variant,
                        "error_type": type(exc).__name__,
                        "message": str(exc),
                    }
                )

        # Estimator-parameter variants operate on the frozen baseline target
        # spectrum and baseline LOSO reference. Each variant is isolated.
        for variant, parameters in parameter_specifications:
            try:
                values = compute_reference_relative_features(
                    baseline_observation,
                    baseline_reference,
                    parameters,
                )
                for feature in ANALYSIS_FEATURES:
                    baseline_value = pd.to_numeric(
                        pd.Series([baseline_values.get(feature)]),
                        errors="coerce",
                    ).iloc[0]
                    variant_value = pd.to_numeric(
                        pd.Series([values.get(feature)]),
                        errors="coerce",
                    ).iloc[0]
                    parameter_sensitivity_rows.append(
                        {
                            "logical_recording_id": recording_id,
                            "variant": variant,
                            "feature": feature,
                            "baseline_value": baseline_value,
                            "variant_value": variant_value,
                            "baseline_available": bool(
                                np.isfinite(baseline_value)
                            ),
                            "variant_available": bool(
                                np.isfinite(variant_value)
                            ),
                            "absolute_delta": (
                                abs(variant_value - baseline_value)
                                if np.isfinite(variant_value)
                                and np.isfinite(baseline_value)
                                else np.nan
                            ),
                        }
                    )
            except Exception as exc:
                target_robustness_errors.append(
                    {
                        "logical_recording_id": recording_id,
                        "stage": "parameter_variant",
                        "variant": variant,
                        "error_type": type(exc).__name__,
                        "message": str(exc),
                    }
                )

# Explicit schemas prevent a secondary KeyError from masking the real failure.
target_robustness = pd.DataFrame(
    target_robustness_rows,
    columns=TARGET_ROBUSTNESS_COLUMNS,
)
parameter_sensitivity = pd.DataFrame(
    parameter_sensitivity_rows,
    columns=PARAMETER_SENSITIVITY_COLUMNS,
)
target_robustness_error_table = pd.DataFrame(
    target_robustness_errors,
    columns=ROBUSTNESS_ERROR_COLUMNS,
)


def summarize_sensitivity(frame, grouping_column):
    summary_columns = [
        grouping_column if column == "variant" else column
        for column in SENSITIVITY_SUMMARY_COLUMNS
    ]
    rows = []
    if frame.empty:
        return pd.DataFrame(columns=summary_columns)
    required_columns = {
        "logical_recording_id",
        grouping_column,
        "feature",
        "baseline_value",
        "variant_value",
        "baseline_available",
        "variant_available",
        "absolute_delta",
    }
    missing_columns = required_columns - set(frame.columns)
    if missing_columns:
        raise RuntimeError(
            "Sensitivity table is missing required columns: "
            f"{sorted(missing_columns)}"
        )
    for keys, local in frame.groupby(
        [grouping_column, "feature"], sort=True
    ):
        delta = pd.to_numeric(
            local["absolute_delta"], errors="coerce"
        ).dropna()
        paired = local.loc[
            local["baseline_available"].astype(bool)
            & local["variant_available"].astype(bool)
        ]
        n, rho = finite_spearman(
            paired["baseline_value"], paired["variant_value"]
        )
        rows.append(
            {
                grouping_column: keys[0],
                "feature": keys[1],
                "recording_count": local[
                    "logical_recording_id"
                ].nunique(),
                "paired_finite_n": n,
                "availability_agreement": float(
                    (
                        local["baseline_available"].astype(bool)
                        == local["variant_available"].astype(bool)
                    ).mean()
                ),
                "spearman_rho": rho,
                "median_absolute_delta": (
                    float(delta.median()) if len(delta) else np.nan
                ),
                "p95_absolute_delta": (
                    float(delta.quantile(0.95))
                    if len(delta)
                    else np.nan
                ),
                "maximum_absolute_delta": (
                    float(delta.max()) if len(delta) else np.nan
                ),
            }
        )
    return pd.DataFrame(rows, columns=summary_columns)


target_robustness_summary = summarize_sensitivity(
    target_robustness, "variant"
)
parameter_sensitivity_summary = summarize_sensitivity(
    parameter_sensitivity, "variant"
)

save_table(
    target_robustness,
    VALIDATION / "qchan_v400_target_robustness_long",
)
save_table(
    target_robustness_summary,
    VALIDATION / "qchan_v400_target_robustness_summary",
)
save_table(
    parameter_sensitivity,
    VALIDATION / "qchan_v400_parameter_sensitivity_long",
)
save_table(
    parameter_sensitivity_summary,
    VALIDATION / "qchan_v400_parameter_sensitivity_summary",
)
save_table(
    target_robustness_error_table,
    AUDIT / "qchan_v400_target_robustness_errors",
    parquet=False,
)

expected_target_variants = {
    "baseline",
    "frame_30ms",
    "frame_50ms",
    "hop_20ms",
    "guard_100ms",
    "guard_300ms",
    "boundary_erosion_50ms",
    "boundary_dilation_50ms",
    "delete_longest_strict_segment",
}
expected_parameter_variants = {
    variant for variant, _ in parameter_sensitivity_specifications()
}
observed_target_variants = set(target_robustness["variant"].dropna())
observed_parameter_variants = set(parameter_sensitivity["variant"].dropna())
completed_target_recordings = target_robustness.loc[
    target_robustness["variant"].eq("baseline"),
    "logical_recording_id",
].nunique()
completed_parameter_recordings = parameter_sensitivity.loc[
    parameter_sensitivity["variant"].eq("baseline"),
    "logical_recording_id",
].nunique()

g6_target_checks = pd.DataFrame(
    [
        {
            "gate": "G6",
            "check": "target robustness sample completed",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_TARGET_ROBUSTNESS
                or completed_target_recordings
                == len(target_robustness_sample)
            ),
            "observed": completed_target_recordings,
            "required": (
                len(target_robustness_sample)
                if RUN_COHORT_EXTRACTION
                else 0
            ),
        },
        {
            "gate": "G6",
            "check": "parameter sensitivity sample completed",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_TARGET_ROBUSTNESS
                or completed_parameter_recordings
                == len(target_robustness_sample)
            ),
            "observed": completed_parameter_recordings,
            "required": (
                len(target_robustness_sample)
                if RUN_COHORT_EXTRACTION
                else 0
            ),
        },
        {
            "gate": "G6",
            "check": "frame/hop/guard/boundary/segment variants characterized",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_TARGET_ROBUSTNESS
                or expected_target_variants.issubset(
                    observed_target_variants
                )
            ),
            "observed": sorted(observed_target_variants),
            "required": sorted(expected_target_variants),
        },
        {
            "gate": "G6",
            "check": "floor/band/rolloff/smoothing/tilt parameters characterized",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_TARGET_ROBUSTNESS
                or expected_parameter_variants.issubset(
                    observed_parameter_variants
                )
            ),
            "observed": sorted(observed_parameter_variants),
            "required": sorted(expected_parameter_variants),
        },
        {
            "gate": "G6",
            "check": "smoothing variants are resolvable on frozen PSD grid",
            "passed": bool(
                "octave_fraction_1" in observed_parameter_variants
                and "octave_fraction_2" in observed_parameter_variants
                and all(
                    parameters.octave_fraction in {1, 2, 3}
                    for _, parameters in parameter_sensitivity_specifications()
                )
            ) if RUN_COHORT_EXTRACTION and RUN_TARGET_ROBUSTNESS else True,
            "observed": {
                "baseline_octave_fraction": PARAMETERS.octave_fraction,
                "alternative_octave_fractions": [1, 2],
                "frequency_bin_spacing_hz": FS / PARAMETERS.n_fft,
            },
            "required": "only technically resolvable smoothing grids",
        },
        {
            "gate": "G6",
            "check": "target robustness errors",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_TARGET_ROBUSTNESS
                or target_robustness_error_table.empty
            ),
            "observed": len(target_robustness_error_table),
            "required": 0,
        },
    ]
)
save_table(
    g6_target_checks,
    VALIDATION / "qchan_v400_g6_target_checks",
    parquet=False,
)
display(g6_target_checks)
display(target_robustness_summary.head(20))
display(parameter_sensitivity_summary.head(20))

if RUN_COHORT_EXTRACTION and RUN_TARGET_ROBUSTNESS:
    if not target_robustness_error_table.empty:
        display(target_robustness_error_table.head(50))
        first = target_robustness_error_table.iloc[0]
        raise RuntimeError(
            "QCHAN G6 robustness produced explicit variant errors. "
            f"First error: recording={first['logical_recording_id']}, "
            f"stage={first['stage']}, variant={first['variant']}, "
            f"{first['error_type']}: {first['message']}"
        )
    if not g6_target_checks["passed"].astype(bool).all():
        raise RuntimeError(
            "QCHAN G6 target/parameter sensitivity checks failed. "
            "Inspect qchan_v400_g6_target_checks.csv."
        )


In [ ]:
# G6/G8 — delete-one-subject, bootstrap, membership-vintage, and weighting audits.
reference_robustness = pd.DataFrame()
reference_robustness_summary = pd.DataFrame()
reference_robustness_errors = []

if (
    RUN_COHORT_EXTRACTION
    and RUN_REFERENCE_ROBUSTNESS
    and len(recording_table)
):
    measured = recording_table.loc[
        recording_table["qchan_family_status"].astype(str).eq("measured"),
        [
            "logical_recording_id",
            "qchan_support_tier",
            "qchan_source_bandwidth_limited",
            *ANALYSIS_FEATURES,
        ],
    ].copy()
    measured["ltas_quantile"] = pd.qcut(
        pd.to_numeric(
            measured["qchan_ltas_distance_db"], errors="coerce"
        ),
        q=min(
            4,
            max(
                1,
                pd.to_numeric(
                    measured["qchan_ltas_distance_db"],
                    errors="coerce",
                ).nunique(),
            ),
        ),
        duplicates="drop",
    ).astype(str)
    reference_robustness_sample = deterministic_stratified_sample(
        measured,
        maximum_rows=REFERENCE_ROBUSTNESS_TARGETS,
        stratum_columns=[
            "qchan_support_tier",
            "qchan_source_bandwidth_limited",
            "ltas_quantile",
        ],
    )
    save_table(
        reference_robustness_sample,
        VALIDATION / "qchan_v400_reference_robustness_sample",
        parquet=False,
    )
    try:
        reference_robustness = reference_robustness_grid(
            reference_robustness_sample,
            spectra=spectra,
            metadata=reference_metadata,
            baseline_references=references,
            bootstrap_iterations=REFERENCE_BOOTSTRAP_ITERATIONS,
            maximum_delete_subjects=MAX_DELETE_REFERENCE_SUBJECTS,
            parameters=PARAMETERS,
        )
        reference_robustness_summary = summarize_reference_robustness(
            reference_robustness
        )
    except Exception as exc:
        reference_robustness_errors.append(
            {
                "error_type": type(exc).__name__,
                "message": str(exc),
            }
        )

reference_robustness_error_table = pd.DataFrame(
    reference_robustness_errors,
    columns=["error_type", "message"],
)
save_table(
    reference_robustness,
    VALIDATION / "qchan_v400_reference_robustness_long",
)
save_table(
    reference_robustness_summary,
    VALIDATION / "qchan_v400_reference_robustness_summary",
)
save_table(
    reference_robustness_error_table,
    AUDIT / "qchan_v400_reference_robustness_errors",
    parquet=False,
)

g6_reference_checks = pd.DataFrame(
    [
        {
            "gate": "G6",
            "check": "reference robustness comparisons complete",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_REFERENCE_ROBUSTNESS
                or {
                    "recording_weighted",
                    "vintage80_1",
                    "vintage80_2",
                    "delete_one_reference_subject",
                    "subject_bootstrap",
                }.issubset(set(reference_robustness["comparison"]))
            ),
            "observed": (
                sorted(reference_robustness["comparison"].unique())
                if len(reference_robustness)
                else []
            ),
            "required": "weighting, two vintages, delete-one, subject bootstrap",
        },
        {
            "gate": "G8",
            "check": "reference robustness summary includes all four features",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_REFERENCE_ROBUSTNESS
                or set(reference_robustness_summary["feature"])
                == set(ANALYSIS_FEATURES)
            ),
            "observed": (
                sorted(reference_robustness_summary["feature"].unique())
                if len(reference_robustness_summary)
                else []
            ),
            "required": list(ANALYSIS_FEATURES),
        },
        {
            "gate": "G6",
            "check": "reference robustness errors",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_REFERENCE_ROBUSTNESS
                or reference_robustness_error_table.empty
            ),
            "observed": len(reference_robustness_error_table),
            "required": 0,
        },
    ]
)
save_table(
    g6_reference_checks,
    VALIDATION / "qchan_v400_g6_reference_checks",
    parquet=False,
)
display(g6_reference_checks)
display(reference_robustness_summary.head(30))


In [ ]:
# G7/G8 — empirical behavior, zero masses, repeats, redundancy, and weighting.
empirical_summary = (
    empirical_feature_summary(recording_table)
    if len(recording_table)
    else pd.DataFrame()
)
status_summary = (
    status_missingness_summary(recording_table)
    if len(recording_table)
    else pd.DataFrame()
)
support_summary = (
    support_availability_summary(recording_table)
    if len(recording_table)
    else pd.DataFrame()
)
bandwidth_summary = (
    native_bandwidth_summary(recording_table)
    if len(recording_table)
    else pd.DataFrame()
)
repeated_summary = (
    repeated_recording_persistence(
        recording_table,
        subject_column=subject_column,
        date_column=date_column,
    )
    if len(recording_table)
    else pd.DataFrame()
)

redundancy_columns = [
    *ANALYSIS_FEATURES,
    *SIGNED_PRECURSORS.values(),
]
redundancy_summary = (
    pairwise_redundancy(recording_table, redundancy_columns)
    if len(recording_table)
    else pd.DataFrame()
)
participant_resampling = (
    participant_balanced_resampling(
        recording_table,
        subject_column=subject_column,
        iterations=PARTICIPANT_BALANCED_ITERATIONS,
    )
    if len(recording_table)
    else pd.DataFrame()
)
participant_summary = participant_balanced_summary(
    participant_resampling
)

recording_vs_participant_rows = []
if len(recording_table):
    for feature in ANALYSIS_FEATURES:
        recording_values = pd.to_numeric(
            recording_table[feature], errors="coerce"
        ).dropna()
        balanced_row = participant_summary.loc[
            participant_summary["feature"].eq(feature)
        ]
        recording_vs_participant_rows.append(
            {
                "feature": feature,
                "recording_weighted_available_n": len(recording_values),
                "recording_weighted_median": (
                    float(recording_values.median())
                    if len(recording_values)
                    else np.nan
                ),
                "participant_balanced_median_of_medians": (
                    float(balanced_row.iloc[0]["median_of_medians"])
                    if len(balanced_row)
                    else np.nan
                ),
                "participant_balanced_p025": (
                    float(balanced_row.iloc[0]["p025_median"])
                    if len(balanced_row)
                    else np.nan
                ),
                "participant_balanced_p975": (
                    float(balanced_row.iloc[0]["p975_median"])
                    if len(balanced_row)
                    else np.nan
                ),
            }
        )
recording_vs_participant = pd.DataFrame(
    recording_vs_participant_rows
)

save_table(
    empirical_summary,
    VALIDATION / "qchan_v400_empirical_feature_summary",
)
save_table(
    status_summary,
    VALIDATION / "qchan_v400_status_missingness_summary",
)
save_table(
    support_summary,
    VALIDATION / "qchan_v400_support_availability",
)
save_table(
    bandwidth_summary,
    VALIDATION / "qchan_v400_native_bandwidth_summary",
)
save_table(
    repeated_summary,
    VALIDATION / "qchan_v400_repeated_recording_persistence",
)
save_table(
    redundancy_summary,
    VALIDATION / "qchan_v400_pairwise_redundancy",
)
save_table(
    participant_resampling,
    VALIDATION / "qchan_v400_participant_balanced_resampling",
)
save_table(
    participant_summary,
    VALIDATION / "qchan_v400_participant_balanced_summary",
)
save_table(
    recording_vs_participant,
    VALIDATION / "qchan_v400_recording_vs_participant_weighting",
)

g7_checks = pd.DataFrame(
    [
        {
            "gate": "G7",
            "check": "exact frozen cohort and participant counts",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(recording_table) == 519
                    and recording_table[
                        subject_column
                    ].nunique(dropna=True)
                    == 224
                )
            ),
            "observed": (
                {
                    "recordings": len(recording_table),
                    "participants": recording_table[
                        subject_column
                    ].nunique(dropna=True),
                }
                if len(recording_table)
                else {}
            ),
            "required": {"recordings": 519, "participants": 224},
        },
        {
            "gate": "G7",
            "check": "all four empirical distributions summarized",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or set(empirical_summary["feature"])
                == set(ANALYSIS_FEATURES)
            ),
            "observed": (
                sorted(empirical_summary["feature"].unique())
                if len(empirical_summary)
                else []
            ),
            "required": list(ANALYSIS_FEATURES),
        },
        {
            "gate": "G7",
            "check": "support and missingness summarized",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(status_summary)
                    and len(support_summary)
                )
            ),
            "observed": {
                "status_rows": len(status_summary),
                "support_rows": len(support_summary),
            },
            "required": "nonempty",
        },
        {
            "gate": "G7",
            "check": "native source bandwidth summarized",
            "passed": bool(
                not RUN_COHORT_EXTRACTION or len(bandwidth_summary)
            ),
            "observed": len(bandwidth_summary),
            "required": ">0",
        },
    ]
)
g8_checks = pd.DataFrame(
    [
        {
            "gate": "G8",
            "check": "feature-specific repeated-recording persistence reported",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or set(repeated_summary["feature"])
                == set(ANALYSIS_FEATURES)
            ),
            "observed": (
                repeated_summary[
                    ["feature", "paired_subject_count"]
                ].to_dict("records")
                if len(repeated_summary)
                else []
            ),
            "required": list(ANALYSIS_FEATURES),
        },
        {
            "gate": "G8",
            "check": "pairwise features and signed precursors reported",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    len(redundancy_summary)
                    and set(redundancy_columns).issubset(
                        set(redundancy_summary["feature_left"])
                        | set(redundancy_summary["feature_right"])
                    )
                )
            ),
            "observed": len(redundancy_summary),
            "required": ">0",
        },
        {
            "gate": "G8",
            "check": "participant-balanced summaries reported",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or set(participant_summary["feature"])
                == set(ANALYSIS_FEATURES)
            ),
            "observed": (
                sorted(participant_summary["feature"].unique())
                if len(participant_summary)
                else []
            ),
            "required": list(ANALYSIS_FEATURES),
        },
        {
            "gate": "G8",
            "check": "reference robustness evidence reported",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not RUN_REFERENCE_ROBUSTNESS
                or len(reference_robustness_summary)
            ),
            "observed": len(reference_robustness_summary),
            "required": ">0",
        },
    ]
)
save_table(
    g7_checks,
    VALIDATION / "qchan_v400_g7_checks",
    parquet=False,
)
save_table(
    g8_checks,
    VALIDATION / "qchan_v400_g8_checks",
    parquet=False,
)
display(g7_checks)
display(g8_checks)
display(empirical_summary)
display(repeated_summary)


In [ ]:
# G10 preparation — support-aware, non-imputed ML handoff.
ml_interface = (
    model_interface_frame(recording_table)
    if len(recording_table)
    else pd.DataFrame()
)
save_table(
    ml_interface,
    TABLES / "qchan_v400_model_ready_features",
)

ml_check_rows = []
for feature in ANALYSIS_FEATURES:
    required_columns = [
        feature,
        f"{feature}__available",
        f"{feature}__status",
        f"{feature}__missing_reason",
        f"{feature}__support_tier",
    ]
    ml_check_rows.append(
        {
            "gate": "G10",
            "check": f"ML interface complete: {feature}",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or all(
                    column in ml_interface
                    for column in required_columns
                )
            ),
            "observed": [
                column
                for column in required_columns
                if column in ml_interface
            ],
            "required": required_columns,
        }
    )
    if len(ml_interface) and feature in ml_interface:
        available = ml_interface[
            f"{feature}__available"
        ].astype(bool)
        missing_values = pd.to_numeric(
            ml_interface.loc[~available, feature],
            errors="coerce",
        )
        ml_check_rows.append(
            {
                "gate": "G10",
                "check": f"no implicit imputation: {feature}",
                "passed": bool(missing_values.isna().all()),
                "observed": int(missing_values.notna().sum()),
                "required": 0,
            }
        )

ml_check_rows.extend(
    [
        {
            "gate": "G10",
            "check": "reference and native-bandwidth context retained",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or all(
                    column in ml_interface
                    for column in [
                        "qchan_reference_sha256",
                        "qchan_reference_vintage_sha256",
                        "qchan_source_sample_rate_hz",
                        "qchan_source_nyquist_hz",
                        "qchan_source_bandwidth_limited",
                    ]
                )
            ),
            "observed": list(ml_interface.columns),
            "required": "reference hashes and native bandwidth fields",
        },
        {
            "gate": "G10",
            "check": "no scalar or standalone gate",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or (
                    not ml_interface[
                        "qchan_family_scalar_available"
                    ]
                    .astype(bool)
                    .any()
                    and not ml_interface[
                        "qchan_standalone_reject_allowed"
                    ]
                    .astype(bool)
                    .any()
                )
            ),
            "observed": {
                "scalar": False,
                "standalone_gate": False,
            },
            "required": {
                "scalar": False,
                "standalone_gate": False,
            },
        },
    ]
)
ml_checks = pd.DataFrame(ml_check_rows)
save_table(
    ml_checks,
    VALIDATION / "qchan_v400_ml_interface_checks",
    parquet=False,
)
display(ml_checks)
display(ml_interface.head())


In [ ]:
# Panels D, E, F, H, and J — cohort publication figures.
if RUN_COHORT_EXTRACTION and len(recording_table):
    # D1 — target support and availability.
    d1 = support_summary.copy()
    d1_plot = d1.pivot(
        index="support_class",
        columns="feature",
        values="availability_fraction_within_class",
    ).reindex(["unavailable", "minimum", "moderate", "high"])
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    x = np.arange(len(d1_plot.index))
    width = 0.18
    for feature_index, feature in enumerate(ANALYSIS_FEATURES):
        ax.bar(
            x + (feature_index - 1.5) * width,
            100 * d1_plot[feature].to_numpy(float),
            width=width,
            label=feature_label(feature),
        )
    ax.set_xticks(x, [str(item) for item in d1_plot.index])
    ax.set_ylabel("Available recordings within support class (%)")
    ax.set_xlabel("Guarded strict-speech support class")
    ax.set_ylim(0, 105)
    ax.set_title("QCHAN target support and feature availability")
    ax.legend(frameon=False, fontsize=8)
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="D1_target_support_availability",
            panel="D1",
            source_data=d1,
            caption=(
                "Availability of each QCHAN feature across guarded strict-speech "
                "support classes. Support class describes evidence quantity and "
                "is not a calibrated quality or precision tier."
            ),
            scientific_question=(
                "How do target support and explicit unavailable states affect "
                "the four reference-relative measurements?"
            ),
            alt_text=(
                "Grouped bar chart of QCHAN feature availability across "
                "unavailable, minimum, moderate, and high strict-speech support classes."
            ),
        )
    )

    # D2 — reference support/status.
    d2 = reference_inventory.copy()
    fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
    measured_reference = d2.loc[
        d2["reference_status"].astype(str).eq("measured")
    ]
    axes[0].hist(
        pd.to_numeric(
            measured_reference["reference_subject_count"],
            errors="coerce",
        ).dropna(),
        bins=20,
    )
    axes[0].axvline(
        PARAMETERS.minimum_reference_subjects,
        linestyle="--",
        linewidth=1.2,
    )
    axes[0].set_xlabel("Other reference participants")
    axes[0].set_ylabel("Target recordings")
    axes[0].set_title("Subject support")
    status_counts = (
        d2["reference_status"].astype(str).value_counts().sort_index()
    )
    axes[1].bar(status_counts.index, status_counts.values)
    axes[1].set_ylabel("Target recordings")
    axes[1].set_title("Reference status")
    axes[1].tick_params(axis="x", rotation=25)
    fig.suptitle("QCHAN task-matched LOSO reference support")
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="D2_reference_support_status",
            panel="D2",
            source_data=d2,
            caption=(
                "Distribution of other-participant support and explicit "
                "reference status for task-matched leave-one-subject-out references. "
                "No global or cross-task fallback is allowed."
            ),
            scientific_question=(
                "Is every target paired with an auditable, adequately supported "
                "task-matched reference?"
            ),
            alt_text=(
                "Histogram of reference participant counts and bar chart of "
                "measured versus unavailable reference states."
            ),
        )
    )

    # D3 — native source bandwidth.
    d3 = bandwidth_summary.copy()
    fig, axes = plt.subplots(1, 2, figsize=(12.0, 4.8))
    axes[0].bar(
        d3["source_sample_rate_hz"].astype(str),
        d3["recording_count"],
    )
    axes[0].set_xlabel("Native source sample rate (Hz)")
    axes[0].set_ylabel("Recordings")
    axes[0].set_title("Native source-rate distribution")
    limited_counts = (
        recording_table["qchan_source_bandwidth_limited"]
        .astype(bool)
        .value_counts()
        .reindex([False, True], fill_value=0)
    )
    axes[1].bar(
        ["full analysis band", "native bandwidth limited"],
        limited_counts.values,
    )
    axes[1].set_ylabel("Recordings")
    axes[1].set_title("Native bandwidth state")
    axes[1].tick_params(axis="x", rotation=15)
    fig.suptitle("QCHAN native source bandwidth provenance")
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="D3_native_bandwidth",
            panel="D3",
            source_data=d3,
            caption=(
                "Native source sample rate, Nyquist support, and explicit "
                "bandwidth-limited state. A limited source can produce apparent "
                "upper-band attenuation and is therefore retained as measurement context."
            ),
            scientific_question=(
                "How much of the cohort enters QCHAN with a native source "
                "bandwidth below the 7.5-kHz analysis ceiling?"
            ),
            alt_text=(
                "Bar charts of native source sample-rate counts and full-band "
                "versus bandwidth-limited recording counts."
            ),
        )
    )

    # E1 — target/window/boundary sensitivity, separate units.
    e1 = target_robustness_summary.loc[
        ~target_robustness_summary["variant"].eq("baseline")
    ].copy()
    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.0))
    for ax, feature in zip(axes.flat, ANALYSIS_FEATURES):
        local = e1.loc[e1["feature"].eq(feature)]
        ax.barh(
            local["variant"],
            local["median_absolute_delta"],
        )
        ax.set_xlabel(f"Median absolute change ({feature_unit(feature)})")
        ax.set_title(feature_label(feature))
    fig.suptitle("QCHAN target window, guard, boundary, and segment sensitivity")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="E1_window_boundary_sensitivity",
            panel="E1",
            source_data=target_robustness_summary,
            caption=(
                "Feature-specific changes under frame, hop, guard, strict-boundary "
                "erosion/dilation, and deterministic segment-deletion variants. "
                "Each feature is shown in its own physical unit."
            ),
            scientific_question=(
                "How strongly do target-side implementation and strict-speech "
                "boundary choices move QCHAN measurements?"
            ),
            alt_text=(
                "Four horizontal bar charts showing median feature changes for "
                "target window, guard, boundary, and segment variants."
            ),
        )
    )

    # E2 — reference robustness, feature-specific.
    e2 = reference_robustness_summary.copy()
    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.0))
    for ax, feature in zip(axes.flat, ANALYSIS_FEATURES):
        local = e2.loc[e2["feature"].eq(feature)]
        ax.barh(
            local["comparison"],
            local["median_absolute_delta"],
        )
        ax.set_xlabel(f"Median absolute change ({feature_unit(feature)})")
        ax.set_title(feature_label(feature))
    fig.suptitle("QCHAN reference-composition robustness")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="E2_reference_robustness",
            panel="E2",
            source_data=reference_robustness_summary,
            caption=(
                "Audit-only comparison of subject-balanced references with "
                "recording-weighted, deterministic membership-vintage, "
                "delete-one-reference-subject, and subject-bootstrap alternatives."
            ),
            scientific_question=(
                "How dependent are the four QCHAN measurements on the frozen "
                "reference membership and weighting rule?"
            ),
            alt_text=(
                "Four horizontal bar charts of feature changes under alternate "
                "reference construction, membership, deletion, and bootstrap audits."
            ),
        )
    )

    # E3 — estimator parameter sensitivity.
    e3 = parameter_sensitivity_summary.loc[
        ~parameter_sensitivity_summary["variant"].eq("baseline")
    ].copy()
    fig, axes = plt.subplots(2, 2, figsize=(13.0, 9.0))
    for ax, feature in zip(axes.flat, ANALYSIS_FEATURES):
        local = e3.loc[e3["feature"].eq(feature)]
        ax.barh(
            local["variant"],
            local["median_absolute_delta"],
        )
        ax.set_xlabel(f"Median absolute change ({feature_unit(feature)})")
        ax.set_title(feature_label(feature))
    fig.suptitle("QCHAN estimator-parameter sensitivity")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="E3_parameter_sensitivity",
            panel="E3",
            source_data=parameter_sensitivity_summary,
            caption=(
                "Feature-specific effects of spectral floor, analysis ceiling, "
                "high-band split, rolloff fraction, octave smoothing, and tilt-band choices."
            ),
            scientific_question=(
                "Which pinned spectral parameters materially define the scale "
                "or ranking of each QCHAN feature?"
            ),
            alt_text=(
                "Four horizontal bar charts of median feature changes under "
                "declared estimator-parameter variants."
            ),
        )
    )

    # F — empirical distributions and one-sided zero masses.
    f_source_rows = []
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
    for ax, feature in zip(axes.flat, ANALYSIS_FEATURES):
        values = pd.to_numeric(
            recording_table[feature], errors="coerce"
        ).dropna()
        ax.hist(values, bins=30)
        if feature in ONE_SIDED_FEATURES:
            ax.axvline(0.0, linestyle="--", linewidth=1.2)
        ax.set_xlabel(feature_unit(feature))
        ax.set_ylabel("Recordings")
        ax.set_title(
            f"{feature_label(feature)} (available n={len(values)})"
        )
        for recording_id, value in recording_table[
            ["logical_recording_id", feature]
        ].itertuples(index=False):
            f_source_rows.append(
                {
                    "logical_recording_id": recording_id,
                    "feature": feature,
                    "value": value,
                    "unit": feature_unit(feature),
                    "available": bool(pd.notna(value)),
                    "is_one_sided_zero": bool(
                        pd.notna(value)
                        and feature in ONE_SIDED_FEATURES
                        and abs(float(value)) <= 1e-12
                    ),
                }
            )
    fig.suptitle("QCHAN empirical distributions and one-sided zero masses")
    fig.tight_layout()
    f_source = pd.DataFrame(f_source_rows)
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="F_empirical_distributions",
            panel="F",
            source_data=f_source,
            caption=(
                "Empirical distributions for the four QCHAN features. "
                "For one-sided features, zero is a measured truncation of a "
                "non-deficit signed difference and is not equivalent to spectral identity."
            ),
            scientific_question=(
                "What are the cohort distributions, tails, availability, and "
                "one-sided zero masses in physical units?"
            ),
            alt_text=(
                "Four histograms showing LTAS distance, rolloff deficit, "
                "high-band deficit, and tilt steepening distributions."
            ),
        )
    )

    # H1 — repeated-recording persistence.
    h1 = repeated_summary.copy()
    fig, ax = plt.subplots(figsize=(10.5, 5.7))
    x = np.arange(len(h1))
    width = 0.35
    ax.bar(
        x - width / 2,
        h1["first_second_spearman"],
        width=width,
        label="First-second Spearman",
    )
    ax.bar(
        x + width / 2,
        h1["icc1_first_two"],
        width=width,
        label="ICC(1), first two",
    )
    ax.set_xticks(
        x,
        [feature_label(feature) for feature in h1["feature"]],
        rotation=15,
    )
    ax.set_ylabel("Reliability coefficient")
    ax.set_ylim(-1, 1)
    ax.axhline(0.0, linewidth=0.8)
    ax.set_title("QCHAN repeated-recording persistence")
    ax.legend(frameon=False)
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="H1_repeated_recordings",
            panel="H1",
            source_data=h1,
            caption=(
                "Feature-specific first-versus-second recording rank persistence "
                "and ICC(1). These statistics describe empirical repeat structure "
                "and do not establish source specificity."
            ),
            scientific_question=(
                "Do QCHAN features show persistent participant-level ordering "
                "across repeated recordings?"
            ),
            alt_text=(
                "Grouped bars of Spearman and ICC reliability coefficients for "
                "the four QCHAN features."
            ),
        )
    )

    # H2 — redundancy matrix with signed precursors.
    h2 = redundancy_summary.copy()
    ordered_columns = redundancy_columns
    matrix = pd.DataFrame(
        np.eye(len(ordered_columns)),
        index=ordered_columns,
        columns=ordered_columns,
    )
    for row in h2.itertuples(index=False):
        matrix.loc[row.feature_left, row.feature_right] = row.spearman_rho
        matrix.loc[row.feature_right, row.feature_left] = row.spearman_rho
    fig, ax = plt.subplots(figsize=(10.5, 9.0))
    image = ax.imshow(matrix.to_numpy(float), vmin=-1, vmax=1, aspect="auto")
    ax.set_xticks(
        np.arange(len(ordered_columns)),
        ordered_columns,
        rotation=70,
        ha="right",
        fontsize=8,
    )
    ax.set_yticks(
        np.arange(len(ordered_columns)),
        ordered_columns,
        fontsize=8,
    )
    ax.set_title("QCHAN rank structure: retained features and signed precursors")
    fig.colorbar(image, ax=ax, label="Spearman rho")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="H2_redundancy",
            panel="H2",
            source_data=h2,
            caption=(
                "Pairwise Spearman rank structure with pairwise sample sizes "
                "for retained features and signed precursors. Correlation does "
                "not justify a family scalar."
            ),
            scientific_question=(
                "Which QCHAN measurements overlap, and how much structure is "
                "introduced by one-sided truncation?"
            ),
            alt_text=(
                "Correlation heatmap for four QCHAN features and three signed "
                "precursor differences."
            ),
        )
    )

    # H3 — participant and reference weighting, separate feature units.
    h3 = recording_vs_participant.copy()
    reference_weighting = reference_robustness_summary.loc[
        reference_robustness_summary["comparison"].eq(
            "recording_weighted"
        ),
        ["feature", "median_absolute_delta", "spearman_rho"],
    ].rename(
        columns={
            "median_absolute_delta": "reference_weighting_median_absolute_delta",
            "spearman_rho": "reference_weighting_spearman_rho",
        }
    )
    h3 = h3.merge(reference_weighting, on="feature", how="left")
    fig, axes = plt.subplots(2, 2, figsize=(12.5, 8.5))
    for ax, feature in zip(axes.flat, ANALYSIS_FEATURES):
        local = h3.loc[h3["feature"].eq(feature)].iloc[0]
        ax.bar(
            ["recording-weighted", "participant-balanced"],
            [
                local["recording_weighted_median"],
                local["participant_balanced_median_of_medians"],
            ],
        )
        ax.set_ylabel(feature_unit(feature))
        ax.set_title(feature_label(feature))
        ax.tick_params(axis="x", rotation=15)
    fig.suptitle("QCHAN recording, participant, and reference weighting")
    fig.tight_layout()
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="H3_weighting",
            panel="H3",
            source_data=h3,
            caption=(
                "Recording-weighted cohort medians compared with participant-balanced "
                "resampling summaries. Source data also report the audit-only effect "
                "of replacing subject-balanced references with recording-weighted references."
            ),
            scientific_question=(
                "Do repeated recordings or unequal reference recording counts "
                "materially change cohort summaries?"
            ),
            alt_text=(
                "Four feature-specific bar charts comparing recording-weighted "
                "and participant-balanced medians."
            ),
        )
    )

    # J — model interface completeness.
    j_rows = []
    for feature in ANALYSIS_FEATURES:
        available = ml_interface[f"{feature}__available"].astype(bool)
        j_rows.extend(
            [
                {
                    "feature": feature,
                    "state": "available",
                    "recording_count": int(available.sum()),
                },
                {
                    "feature": feature,
                    "state": "unavailable",
                    "recording_count": int((~available).sum()),
                },
            ]
        )
    j_source = pd.DataFrame(j_rows)
    pivot = j_source.pivot(
        index="feature",
        columns="state",
        values="recording_count",
    ).reindex(ANALYSIS_FEATURES)
    fig, ax = plt.subplots(figsize=(10.5, 5.8))
    y = np.arange(len(pivot))
    ax.barh(y, pivot["available"], label="available")
    ax.barh(
        y,
        pivot["unavailable"],
        left=pivot["available"],
        label="unavailable",
    )
    ax.set_yticks(y, [feature_label(feature) for feature in pivot.index])
    ax.set_xlabel("Recordings")
    ax.set_title("QCHAN support-aware ML interface")
    ax.legend(frameon=False)
    figure_index_rows.append(
        save_figure_bundle(
            fig,
            stem="J_ml_interface",
            panel="J",
            source_data=j_source,
            caption=(
                "Availability states in the non-imputed QCHAN ML handoff. "
                "Every value is accompanied by status, support, native bandwidth, "
                "reference counts, and reference hashes; no family scalar is present."
            ),
            scientific_question=(
                "Does the downstream interface preserve measurement availability "
                "and provenance without implicit imputation?"
            ),
            alt_text=(
                "Stacked horizontal bars showing available and unavailable "
                "recordings for each QCHAN feature."
            ),
        )
    )


In [ ]:
# Panel G — deterministic label-blind waveform/spectrogram/LTAS/reference examples.
gallery_rows = []
gallery_errors = []

if (
    RUN_COHORT_EXTRACTION
    and BUILD_GALLERY
    and len(recording_table)
):
    working = recording_table.copy()
    selection = deterministic_gallery_selection(
        working,
        maximum_rows=GALLERY_RECORDING_LIMIT,
        minimum_rows=8,
    )
    save_table(
        selection,
        GALLERIES / "qchan_v400_gallery_selection",
        parquet=False,
    )
    frozen_lookup = frozen.set_index("logical_recording_id")
    feature_lookup = recording_features.set_index("logical_recording_id")

    for selected in selection.itertuples(index=False):
        recording_id = str(selected.logical_recording_id)
        try:
            source_row = frozen_lookup.loc[recording_id]
            media_path = resolve_media_path(
                source_row["media_path"],
                media_root_override=(
                    Path(MEDIA_ROOT_OVERRIDE)
                    if MEDIA_ROOT_OVERRIDE
                    else None
                ),
                media_path_map=MEDIA_PATH_MAP,
            )
            views = decode_audio_views(
                media_path,
                ffmpeg=ffmpeg,
                ffprobe=ffprobe,
                analysis_rate=FS,
            )
            waveform, _, _ = remove_global_dc(
                analysis_waveform_from_audio_views(views)
            )
            strict, _ = intervals_for_recording(
                strict_table, recording_id
            )
            if strict:
                chosen_interval = max(
                    strict,
                    key=lambda item: item.end_sec - item.start_sec,
                )
                center = 0.5 * (
                    chosen_interval.start_sec + chosen_interval.end_sec
                )
                plot_start = max(0.0, center - 1.5)
                plot_end = min(
                    len(waveform) / FS,
                    center + 1.5,
                )
            else:
                plot_start = 0.0
                plot_end = min(3.0, len(waveform) / FS)

            left = int(np.floor(plot_start * FS))
            right = int(np.ceil(plot_end * FS))
            audio = waveform[left:right]
            time_axis = np.arange(left, right, dtype=float) / FS
            waveform_stride = max(
                1, int(np.ceil(max(len(audio), 1) / 5000))
            )
            wave_source = pd.DataFrame(
                {
                    "row_type": "waveform",
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "selection_source": selected.selection_source,
                    "selection_order": int(selected.selection_order),
                    "time_sec": time_axis[::waveform_stride],
                    "frequency_hz": np.nan,
                    "amplitude": audio[::waveform_stride],
                    "power_db": np.nan,
                    "observation_ltas_db": np.nan,
                    "reference_ltas_db": np.nan,
                    "difference_db": np.nan,
                }
            )

            nperseg = int(min(512, max(64, len(audio))))
            noverlap = int(
                min(nperseg - 1, round(0.75 * nperseg))
            )
            frequencies, spec_times, spectrum_matrix = signal.spectrogram(
                audio,
                fs=FS,
                window="hann",
                nperseg=nperseg,
                noverlap=noverlap,
                detrend="constant",
                scaling="spectrum",
                mode="psd",
            )
            power_db = 10.0 * np.log10(
                np.maximum(
                    spectrum_matrix,
                    np.finfo(float).tiny,
                )
            )
            spec_source = pd.DataFrame(
                {
                    "row_type": "spectrogram",
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "time_sec": np.tile(
                        spec_times + plot_start,
                        len(frequencies),
                    ),
                    "frequency_hz": np.repeat(
                        frequencies,
                        len(spec_times),
                    ),
                    "amplitude": np.nan,
                    "power_db": power_db.reshape(-1),
                    "observation_ltas_db": np.nan,
                    "reference_ltas_db": np.nan,
                    "difference_db": np.nan,
                }
            )

            ltas = ltas_source_frame(
                spectra[recording_id],
                references[recording_id],
                PARAMETERS,
            )
            if len(ltas):
                ltas_source = pd.DataFrame(
                    {
                        "row_type": "ltas",
                        "logical_recording_id": recording_id,
                        "selection_reason": selected.selection_reason,
                        "time_sec": np.nan,
                        "frequency_hz": ltas[
                            "center_frequency_hz"
                        ],
                        "amplitude": np.nan,
                        "power_db": np.nan,
                        "observation_ltas_db": ltas[
                            "observation_ltas_db"
                        ],
                        "reference_ltas_db": ltas[
                            "reference_ltas_db"
                        ],
                        "difference_db": ltas["difference_db"],
                    }
                )
            else:
                ltas_source = pd.DataFrame(
                    columns=wave_source.columns
                )

            feature_row = feature_lookup.loc[recording_id]
            feature_source = pd.DataFrame(
                [
                    {
                        "row_type": "features",
                        "logical_recording_id": recording_id,
                        "selection_reason": selected.selection_reason,
                        "time_sec": np.nan,
                        "frequency_hz": np.nan,
                        "amplitude": np.nan,
                        "power_db": np.nan,
                        "observation_ltas_db": np.nan,
                        "reference_ltas_db": np.nan,
                        "difference_db": np.nan,
                        **{
                            feature: feature_row.get(feature)
                            for feature in ANALYSIS_FEATURES
                        },
                        "qchan_family_status": feature_row.get(
                            "qchan_family_status"
                        ),
                        "qchan_support_tier": feature_row.get(
                            "qchan_support_tier"
                        ),
                        "qchan_reference_subject_count": feature_row.get(
                            "qchan_reference_subject_count"
                        ),
                        "qchan_source_sample_rate_hz": feature_row.get(
                            "qchan_source_sample_rate_hz"
                        ),
                        "qchan_source_bandwidth_limited": feature_row.get(
                            "qchan_source_bandwidth_limited"
                        ),
                    }
                ]
            )
            source = gallery_linked_view_source(
                wave_source,
                spec_source,
                ltas_source,
                feature_source,
                unavailable_reason=(
                    "reference_relative_ltas_unavailable_under_declared_support_contract"
                ),
            )
            source["source_media_sha256"] = sha256_file(media_path)

            fig, axes = plt.subplots(
                2, 2, figsize=(12.5, 8.5)
            )
            axes[0, 0].plot(
                time_axis[::waveform_stride],
                audio[::waveform_stride],
                linewidth=0.7,
            )
            for interval in strict:
                if (
                    interval.end_sec >= plot_start
                    and interval.start_sec <= plot_end
                ):
                    axes[0, 0].axvspan(
                        max(plot_start, interval.start_sec),
                        min(plot_end, interval.end_sec),
                        alpha=0.12,
                    )
            axes[0, 0].set_xlabel("Time (s)")
            axes[0, 0].set_ylabel("Amplitude")
            axes[0, 0].set_title("Guarded strict-speech context")

            axes[0, 1].pcolormesh(
                spec_times + plot_start,
                frequencies,
                power_db,
                shading="auto",
            )
            axes[0, 1].set_ylim(0, 8000)
            axes[0, 1].set_xlabel("Time (s)")
            axes[0, 1].set_ylabel("Frequency (Hz)")
            axes[0, 1].set_title("Spectrogram")

            if len(ltas):
                axes[1, 0].semilogx(
                    ltas["center_frequency_hz"],
                    ltas["observation_ltas_db"],
                    marker="o",
                    label="target",
                )
                axes[1, 0].semilogx(
                    ltas["center_frequency_hz"],
                    ltas["reference_ltas_db"],
                    marker="o",
                    label="LOSO reference",
                )
                axes[1, 0].set_xlabel("Center frequency (Hz)")
                axes[1, 0].set_ylabel("Gain-normalized LTAS (dB)")
                axes[1, 0].legend(frameon=False)
                axes[1, 0].set_title("Target and frozen reference")
                axes[1, 1].semilogx(
                    ltas["center_frequency_hz"],
                    ltas["difference_db"],
                    marker="o",
                )
                axes[1, 1].axhline(0.0, linewidth=0.8)
                axes[1, 1].set_xlabel("Center frequency (Hz)")
                axes[1, 1].set_ylabel("Target minus reference (dB)")
                axes[1, 1].set_title("Reference-relative spectral difference")
            else:
                for ax in [axes[1, 0], axes[1, 1]]:
                    ax.text(
                        0.5,
                        0.5,
                        "Reference-relative LTAS unavailable\nunder the declared support contract",
                        ha="center",
                        va="center",
                        transform=ax.transAxes,
                    )
                    ax.set_axis_off()

            fig.suptitle(
                f"QCHAN signal example: {selected.selection_reason}\n"
                f"{recording_id}"
            )
            fig.tight_layout()

            safe_id = "".join(
                character
                if character.isalnum() or character in "._-"
                else "_"
                for character in recording_id
            )
            stem = f"G_{selected.selection_reason}_{safe_id}"
            bundle = save_figure_bundle(
                fig,
                stem=stem,
                panel="G",
                source_data=source,
                destination=GALLERIES,
                caption=(
                    f"Deterministic label-blind QCHAN example selected as "
                    f"`{selected.selection_reason}`. Waveform and spectrogram "
                    f"show the signal context; target/reference LTAS and their "
                    f"difference show the measured spectral manifestation. "
                    f"No device or causal source label is assigned."
                ),
                scientific_question=(
                    "Do real signal examples visually support the measured "
                    "reference-relative spectral patterns and unavailable states?"
                ),
                alt_text=(
                    f"Four-panel QCHAN example for {recording_id}: waveform, "
                    "spectrogram, target versus reference LTAS, and spectral difference."
                ),
                provenance={
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "selection_source": selected.selection_source,
                    "selection_order": int(selected.selection_order),
                    "gallery_selection_policy": "unique_reason_prioritized_deterministic_fill_v1",
                    "gallery_source_contract": "five_explicit_linked_views_v1",
                    "source_media_sha256": sha256_file(media_path),
                    "diagnosis_used_for_selection": False,
                    "human_qc_used_for_selection": False,
                },
            )
            linked_views = sorted(set(source["view"].astype(str)))
            gallery_rows.append(
                {
                    **bundle,
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "linked_views": "|".join(linked_views),
                    "linked_view_count": len(linked_views),
                    "linked_views_complete": bool(
                        set(REQUIRED_GALLERY_LINKED_VIEWS).issubset(
                            set(linked_views)
                        )
                    ),
                }
            )
        except Exception as exc:
            gallery_errors.append(
                {
                    "logical_recording_id": recording_id,
                    "selection_reason": selected.selection_reason,
                    "error_type": type(exc).__name__,
                    "message": str(exc),
                }
            )

gallery_index = pd.DataFrame(gallery_rows)
gallery_error_table = pd.DataFrame(
    gallery_errors,
    columns=[
        "logical_recording_id",
        "selection_reason",
        "error_type",
        "message",
    ],
)
save_table(
    gallery_index,
    GALLERIES / "qchan_v400_gallery_index",
    parquet=False,
)
save_table(
    gallery_error_table,
    AUDIT / "qchan_v400_gallery_errors",
    parquet=False,
)

gallery_checks = pd.DataFrame(
    [
        {
            "gate": "G7",
            "check": "at least eight deterministic signal examples",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not BUILD_GALLERY
                or len(gallery_index) >= 8
            ),
            "observed": len(gallery_index),
            "required": ">=8",
        },
        {
            "gate": "G7",
            "check": "gallery recording identities are unique",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not BUILD_GALLERY
                or (
                    len(gallery_index)
                    == gallery_index.get(
                        "logical_recording_id",
                        pd.Series(dtype=str),
                    ).nunique()
                )
            ),
            "observed": (
                gallery_index.get(
                    "logical_recording_id",
                    pd.Series(dtype=str),
                ).nunique()
            ),
            "required": "equal to gallery row count",
        },
        {
            "gate": "G7",
            "check": "each gallery source declares five linked views",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not BUILD_GALLERY
                or (
                    len(gallery_index)
                    and gallery_index.get(
                        "linked_views_complete",
                        pd.Series(dtype=bool),
                    ).astype(bool).all()
                )
            ),
            "observed": (
                int(
                    gallery_index.get(
                        "linked_views_complete",
                        pd.Series(dtype=bool),
                    ).astype(bool).sum()
                )
                if len(gallery_index)
                else 0
            ),
            "required": "all gallery examples",
        },
        {
            "gate": "G7",
            "check": "gallery generation errors",
            "passed": bool(
                not RUN_COHORT_EXTRACTION
                or not BUILD_GALLERY
                or gallery_error_table.empty
            ),
            "observed": len(gallery_error_table),
            "required": 0,
        },
        {
            "gate": "G7",
            "check": "gallery selection excludes diagnosis and human QC",
            "passed": True,
            "observed": "signal-derived deterministic strata only",
            "required": "no diagnosis/human-QC selection",
        },
    ]
)
save_table(
    gallery_checks,
    VALIDATION / "qchan_v400_gallery_checks",
    parquet=False,
)
display(gallery_checks)
display(gallery_index)


In [ ]:
# Complete the cohort checklist evidence package and candidate manifest.
main_figure_index = pd.DataFrame(figure_index_rows)
if len(gallery_index):
    combined_figure_index = pd.concat(
        [main_figure_index, gallery_index],
        ignore_index=True,
        sort=False,
    )
else:
    combined_figure_index = main_figure_index.copy()

panel_i_record = pd.DataFrame(
    [
        {
            "panel": "I",
            "stem": "I_NA_no_retained_event_detector",
            "status": "N/A",
            "reason": "QCHAN contains no retained discrete event detector.",
        }
    ]
)
save_table(
    panel_i_record,
    VALIDATION / "qchan_v400_panel_i_na",
    parquet=False,
)
save_table(
    combined_figure_index,
    TABLES / "qchan_v400_figure_index",
    parquet=False,
)

required_main_panels = {
    "A",
    "B",
    "C",
    "D1",
    "D2",
    "D3",
    "E1",
    "E2",
    "E3",
    "F",
    "H1",
    "H2",
    "H3",
    "J",
}
present_main_panels = set(
    main_figure_index.get("panel", pd.Series(dtype=str)).astype(str)
)
required_main_complete = bool(
    not RUN_COHORT_EXTRACTION
    or required_main_panels.issubset(present_main_panels)
)
gallery_complete = bool(
    not RUN_COHORT_EXTRACTION
    or not BUILD_GALLERY
    or (
        len(gallery_index) >= 8
        and len(gallery_index)
        == gallery_index.get(
            "logical_recording_id",
            pd.Series(dtype=str),
        ).nunique()
        and gallery_index.get(
            "linked_views_complete",
            pd.Series(dtype=bool),
        ).astype(bool).all()
    )
)

bundle_contract_rows = []
if RUN_COHORT_EXTRACTION:
    for row in combined_figure_index.itertuples(index=False):
        paths = [
            STAGE / str(getattr(row, field))
            for field in [
                "png",
                "svg",
                "pdf",
                "source_csv",
                "caption",
                "provenance",
            ]
        ]
        bundle_contract_rows.append(
            {
                "panel": getattr(row, "panel"),
                "stem": getattr(row, "stem"),
                "bundle_complete": all(path.exists() for path in paths),
                "artifact_count": sum(path.exists() for path in paths),
            }
        )
figure_contract = pd.DataFrame(bundle_contract_rows)
save_table(
    figure_contract,
    VALIDATION / "qchan_v400_figure_contract",
    parquet=False,
)

feature_decisions = pd.DataFrame(
    [
        {
            "feature": "qchan_ltas_distance_db",
            "provisional_role": "primary nonordinal spectral-deviation candidate",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "RMS target-versus-frozen-reference LTAS deviation; not intrinsically worse and not a device identity",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qchan_rolloff95_deficit_hz",
            "provisional_role": "primary one-sided upper-extent candidate",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "one-sided reduction in 95%-power spectral rolloff with signed precursor retained",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qchan_highband_ratio_deficit",
            "provisional_role": "secondary one-sided high-band candidate",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "one-sided reduction in 3-7.5-kHz power share with signed precursor retained",
            "standalone_gate_allowed": False,
        },
        {
            "feature": "qchan_tilt_steepening_db_per_oct",
            "provisional_role": "secondary phenotype-sensitive tilt candidate",
            "g10_decision": "PENDING_COHORT_REVIEW",
            "claim": "one-sided spectral-tilt steepening; not independent of voice physiology or phonetic composition",
            "standalone_gate_allowed": False,
        },
    ]
)
save_table(
    feature_decisions,
    VALIDATION / "qchan_v400_g10_feature_decisions",
    parquet=False,
)

check_tables = [
    input_checks,
    extraction_checks,
    reference_checks,
    g2_checks,
    g6_target_checks,
    g6_reference_checks,
    g7_checks,
    g8_checks,
    ml_checks,
    gallery_checks,
]
cohort_checks = pd.concat(
    check_tables,
    ignore_index=True,
    sort=False,
)
save_table(
    cohort_checks,
    VALIDATION / "qchan_v400_cohort_checks",
    parquet=False,
)

cohort_evidence_complete = bool(
    RUN_COHORT_EXTRACTION
    and cohort_checks["passed"].astype(bool).all()
    and required_main_complete
    and gallery_complete
    and (
        figure_contract.empty
        or figure_contract["bundle_complete"].astype(bool).all()
    )
)

gate_summary_cohort = pd.DataFrame(
    [
        {"gate": "G1", "status": "PASS", "scope": "contract and frozen provenance"},
        {"gate": "G2", "status": "PASS", "scope": "numerical reconstruction and missing-value contract"},
        {"gate": "G3", "status": "PASS", "scope": "analysis view, native geometry, transformations and codecs"},
        {"gate": "G4", "status": "PASS", "scope": "controlled construct response"},
        {"gate": "G5", "status": "CONDITIONAL", "scope": "irreducible source/phenotype/noise non-identifiability"},
        {"gate": "G6", "status": "EVIDENCE_COMPLETE_PENDING_REVIEW", "scope": "target, parameter and reference sensitivity"},
        {"gate": "G7", "status": "EVIDENCE_COMPLETE_PENDING_REVIEW", "scope": "empirical cohort, support, bandwidth and gallery"},
        {"gate": "G8", "status": "EVIDENCE_COMPLETE_PENDING_REVIEW", "scope": "repeats, redundancy, participant/reference weighting"},
        {"gate": "G9", "status": "N/A", "scope": "no retained event detector"},
        {"gate": "G10", "status": "PENDING", "scope": "final feature roles and immutable freezes"},
    ]
)
save_table(
    gate_summary_cohort,
    VALIDATION / "qchan_v400_gate_summary_cohort",
    parquet=False,
)

checklist_path = (
    ROOT
    / "docs reviewed"
    / "QCHAN_Validation_Checklist_v0_3_COHORT_READY.csv"
)
if checklist_path.exists():
    checklist = pd.read_csv(checklist_path)
    cohort_pass_items = {
        "G2.3",
        "G3.5",
        "G6.6",
        "G6.7",
        "G7.1",
        "G7.2",
        "G7.3",
        "G7.4",
        "G7.5",
        "G7.6",
        "G8.1",
        "G8.2",
        "G8.3",
        "G8.4",
        "G8.5",
        "G8.6",
        "G10.3",
    }
    checklist.loc[
        checklist["item_id"].isin(cohort_pass_items),
        "status",
    ] = np.where(
        cohort_evidence_complete,
        "EVIDENCE_COMPLETE_PENDING_REVIEW",
        "COHORT_RUN_INCOMPLETE",
    )
    checklist.loc[
        checklist["item_id"].isin(cohort_pass_items),
        "evidence",
    ] = "QCHAN v4.0.0 cohort candidate tables, ledgers, validations and figure bundles"
    checklist.loc[
        checklist["item_id"].isin(["G9.1", "G9.2"]),
        "status",
    ] = "N/A"
else:
    checklist = pd.DataFrame()
save_table(
    checklist,
    VALIDATION / "qchan_v400_checklist_cohort",
    parquet=False,
)

candidate_inventory = hash_inventory(STAGE)
save_table(
    candidate_inventory,
    MANIFESTS / "qchan_v400_candidate_artifact_inventory",
    parquet=False,
)

manifest = {
    "measurement_version": MEASUREMENT_VERSION,
    "candidate_directory": CANDIDATE_DIRNAME,
    "cohort_orchestration_version": COHORT_ORCHESTRATION_VERSION,
    "candidate_only": True,
    "accepted_preflight": accepted_preflight,
    "preflight_blocking_checks_pass": bool(
        preflight_manifest["preflight_blocking_checks_pass"]
    ),
    "package_tests_passed": bool(package_tests_passed),
    "cohort_extraction_completed": bool(
        RUN_COHORT_EXTRACTION
        and len(recording_table) == len(frozen)
    ),
    "cohort_evidence_complete": cohort_evidence_complete,
    "recording_count": int(len(recording_table)),
    "participant_count": int(
        recording_table[subject_column].nunique(dropna=True)
        if len(recording_table)
        else 0
    ),
    "spectrum_count": int(len(spectra)),
    "reference_ledger_count": int(len(reference_inventory)),
    "unique_reference_count": int(len(unique_references)),
    "reference_vintage_count": int(
        reference_inventory[
            "reference_vintage_sha256"
        ].nunique()
        if len(reference_inventory)
        else 0
    ),
    "extraction_error_count": int(len(extraction_errors)),
    "reference_error_count": int(len(reference_error_table)),
    "target_robustness_error_count": int(
        len(target_robustness_error_table)
    ),
    "reference_robustness_error_count": int(
        len(reference_robustness_error_table)
    ),
    "gallery_error_count": int(len(gallery_error_table)),
    "gallery_selection_policy": "unique_reason_prioritized_deterministic_fill_v1",
    "gallery_source_contract": "five_explicit_linked_views_v1",
    "gallery_minimum_required": 8,
    "cohort_hotfix_revision": "gallery-linked-source-r3",
    "analysis_waveform_field": "analysis_16k",
    "global_dc_removal_enforced": True,
    "task_stratum_source": task_source,
    "reference_rule": "task-matched subject-balanced leave-one-subject-out; no fallback",
    "required_panels_complete": bool(
        required_main_complete and gallery_complete
    ),
    "main_figure_bundle_count": int(len(main_figure_index)),
    "gallery_bundle_count": int(len(gallery_index)),
    "panel_i_status": "N/A_no_retained_event_detector",
    "feature_values_recomputed_by_figures": False,
    "family_scalar_constructed": False,
    "standalone_gate_allowed": False,
    "device_identity_estimated": False,
    "scientific_review_decision": SCIENTIFIC_REVIEW_DECISION,
    "freeze_allowed": False,
    "publish_and_freeze": bool(PUBLISH_AND_FREEZE),
    "created_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(
    manifest,
    MANIFESTS / "qchan_v400_cohort_candidate_manifest.json",
)

print("QCHAN v4.0.0 REVIEWED COHORT RUN COMPLETE")
print(json.dumps(manifest, indent=2))
display(gate_summary_cohort)
display(feature_decisions)


## Required next action

Save this fully executed notebook, close JupyterLab, and package the notebook plus the complete `MAIN outputs/02_FEATURE_REVIEWED/00_working_candidates/channel_device/qchan-v4.0.0-candidate` directory for independent post-cohort scientific review.

The next review will audit G6–G8 evidence, every A–H/J figure bundle, reference-vintage sensitivity, one-sided zero masses, repeated-recording behavior, final feature roles, and freeze authorization. Do not publish or freeze QCHAN from this notebook.
